# AMRfold: Structure-informed AMR Mining in *Neisseria gonorrhoeae*
## Complete Analysis Pipeline v3.1

**Author**: Pranavathiyani G | SASTRA Deemed University | April 2026  
**Runtime**: Google Colab T4 GPU | **Seed**: 142857  
**Requirement**: Runtime → Change runtime type → **T4 GPU**

---
### Changelog v3.0 (from v2.2)
- **Query DB**: CARD protein homolog + UniProt SwissProt KW-0046 bacteria → 90% NR (protein only, MEGARes dropped — nucleotide incompatible)
- **5th method**: AMRFinderPlus HMM-based search (binary install, no conda, no kernel restart)
- **ESM2**: Statistically calibrated threshold via NEIG1-NEIG1 null distribution (p=1e-5), replaces arbitrary 0.85
- **UMAP**: AMR protein embedding space analysis + NEIG1 hits overlaid
- **Multi-domain**: Position-aware non-overlapping AMR domain detection per NEIG1 protein
- **Coverage-identity heatmap**: Novel analysis of structural hit categories
- **PDB validation**: 25 representative AMR structures + Foldseek TM-score
- **Checkpoints**: Google Drive checkpoints after every expensive step — session-crash safe
- **Figures**: Individual 300 DPI PNGs, colorblind-safe, panel letters, proper annotation
- **GO enrichment**: Fisher's exact + BH FDR via GOATOOLS (Klopfenstein et al. Sci Reports 2018)
- **KEGG**: UniProt cross-ref mapping + pathway completeness + enrichment (proper NGO IDs)
- **Background**: Full NEIG1 proteome (2,106 proteins) as GO enrichment background
- **Report**: Comprehensive HTML + PDF with all results per step

### Session strategy
This run takes ~140 min. Use Drive checkpoints to split across sessions:
- Session 1: Cells 0-5a (setup + Foldseek, ~70 min)
- Session 2: Load checkpoint, Cells 5b-20 (~70 min)


## Cell 0: Google Drive Mount + Configuration

In [ ]:
# ── CELL 0: Drive Mount + CONFIG ─────────────────────────────────────────────
# Mount first — all checkpoints save here
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── USER CONFIGURATION ─────────────────────────────────────────────────────
# Change these to reuse for a different organism
CONFIG = {
    # Target organism
    'organism_name':     'Neisseria gonorrhoeae FA 1090',
    'uniprot_proteome':  'UP000000535',
    'afdb_tar_url':      'https://ftp.ebi.ac.uk/pub/databases/alphafold/v6/UP000000535_242231_NEIG1_v6.tar',
    'ncbi_organism_flag': 'Neisseria',   # for AMRFinderPlus --organism

    # Query databases
    'card_version':      '4.0.1',
    'card_url':          'https://card.mcmaster.ca/download/0/broadstreet-v4.0.1.tar.bz2',
    'swissprot_amr_url': 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=keyword:KW-0046+reviewed:true+taxonomy_id:2',

    # Search parameters
    'foldseek_sensitivity': 9.5,   # maximum sensitivity for paper
    'evalue_threshold':     1e-5,
    'qcov_min':             0.5,
    'tcov_min':             0.3,
    'cluster_identity':     0.90,
    'pident_cryptic':       30,

    # ESM2 statistical calibration
    'esm2_null_pairs':      10000,
    'esm2_pvalue':          1e-5,
    'esm2_model':           'facebook/esm2_t33_650M_UR50D',

    # Reproducibility
    'random_seed':          142857,   # cyclic number: 1/7 = 0.142857...

    # Paths
    'drive_checkpoint_dir': '/content/drive/MyDrive/amrfold_v3_checkpoints',
    'local_data_dir':       '/content/data',
    'local_results_dir':    '/content/results',
    'local_figures_dir':    '/content/results/figures',
    'local_tmp_dir':        '/content/tmp',
}

# Create checkpoint dir on Drive
Path(CONFIG['drive_checkpoint_dir']).mkdir(parents=True, exist_ok=True)

# Create local dirs
for key in ['local_data_dir', 'local_results_dir', 'local_figures_dir', 'local_tmp_dir']:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)

# Separate tmp dirs per tool — prevents race conditions
for tool in ['foldseek', 'mmseqs', 'diamond', 'human', 'cluster', 'prostt5']:
    Path(f"{CONFIG['local_tmp_dir']}/{tool}").mkdir(parents=True, exist_ok=True)

# Sub-data dirs
for subdir in ['card', 'swissprot', 'neisseria', 'dbs', 'human', 'card_pdb', 'amrfinder']:
    Path(f"{CONFIG['local_data_dir']}/{subdir}").mkdir(parents=True, exist_ok=True)

print('✓ Drive mounted')
print(f'✓ Checkpoint dir: {CONFIG["drive_checkpoint_dir"]}')
print(f'✓ Seed: {CONFIG["random_seed"]} (cyclic number 1/7)')
print(f'✓ Config locked: {len(CONFIG)} parameters')

# ── Checkpoint utilities ───────────────────────────────────────────────────
import shutil

def ckpt_path(name):
    return Path(CONFIG['drive_checkpoint_dir']) / name

def checkpoint_exists(name):
    return ckpt_path(name).exists()

def save_checkpoint(local_path, name):
    shutil.copy(local_path, ckpt_path(name))
    size = ckpt_path(name).stat().st_size / 1e6
    print(f'  ✓ Checkpoint saved: {name} ({size:.1f} MB)')

def load_checkpoint(name, local_path):
    shutil.copy(ckpt_path(name), local_path)
    size = Path(local_path).stat().st_size / 1e6
    print(f'  ✓ Checkpoint loaded: {name} ({size:.1f} MB)')

def checkpoint_status():
    checkpoints = ['foldseek_raw.tsv', 'mmseqs_raw.tsv', 'diamond_raw.tsv',
                   'human_raw.tsv', 'amrfinder_out.tsv', 'esm2_hits.json',
                   'uniprot_cache.json', 'combined_amr_nr.fasta',
                   'cluster_membership.tsv']
    print('\nCheckpoint status:')
    for cp in checkpoints:
        exists = checkpoint_exists(cp)
        size = f'({ckpt_path(cp).stat().st_size/1e6:.1f} MB)' if exists else ''
        print(f'  {"✓" if exists else "○"} {cp} {size}')

checkpoint_status()


## Cell 1: Environment Setup

In [ ]:
# ── CELL 1: Environment setup ─────────────────────────────────────────────────
# ~5 min — installs Foldseek, MMseqs2, DIAMOND, AMRFinderPlus binary
import os, subprocess, threading, time, gzip, csv, json, shutil, random
import numpy as np
from datetime import datetime
from pathlib import Path

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = CONFIG['random_seed']
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed set: {SEED}')

!nvidia-smi | grep -E 'T4|A100|V100|Memory-Usage'

# ── Parallel tool installs ─────────────────────────────────────────────────
def install(cmd, label):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True)
    elapsed = time.time() - t0
    status = 'OK' if r.returncode == 0 else 'FAILED'
    print(f'  {label}: {status} ({elapsed:.0f}s)')
    return r.returncode == 0

print('Installing tools in parallel...')
tool_cmds = [
    ('wget -q https://mmseqs.com/foldseek/foldseek-linux-gpu.tar.gz && tar -xzf foldseek-linux-gpu.tar.gz', 'Foldseek GPU'),
    ('wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz && tar -xzf mmseqs-linux-avx2.tar.gz', 'MMseqs2'),
    ('wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.8/diamond-linux64.tar.gz && tar -xzf diamond-linux64.tar.gz', 'DIAMOND'),
]
threads = [threading.Thread(target=install, args=(cmd, lbl)) for cmd, lbl in tool_cmds]
for t in threads: t.start()
for t in threads: t.join()

# ── AMRFinderPlus binary (no conda — avoids kernel restart) ───────────────
print('\nInstalling AMRFinderPlus binary...')
t0 = time.time()
!apt-get install -y -qq blast+ hmmer 2>/dev/null
amr_url = subprocess.run(
    'curl -s https://api.github.com/repos/ncbi/amr/releases/latest '
    '| grep "browser_download_url.*amrfinder_binaries" | cut -d '"' -f 4',
    shell=True, capture_output=True, text=True).stdout.strip()
subprocess.run(f'curl -sOL "{amr_url}"', shell=True)
tarfile = Path(amr_url).name
subprocess.run(f'tar xfz {tarfile} -C /content/', shell=True)
# Find amrfinder binary
amrfinder_bin = subprocess.run('find /content -name "amrfinder" -type f 2>/dev/null | head -1',
                                shell=True, capture_output=True, text=True).stdout.strip()
print(f'  AMRFinderPlus binary: {amrfinder_bin} ({time.time()-t0:.0f}s)')

# Download AMRFinderPlus database
print('  Downloading AMRFinderPlus database (~300MB)...')
t0 = time.time()
subprocess.run(f'{amrfinder_bin} -u --database {CONFIG["local_data_dir"]}/amrfinder/db',
               shell=True, capture_output=True)
amrf_db_date = subprocess.run(
    f'cat {CONFIG["local_data_dir"]}/amrfinder/db/version.txt 2>/dev/null || echo unknown',
    shell=True, capture_output=True, text=True).stdout.strip()
print(f'  AMRFinderPlus DB date: {amrf_db_date} ({time.time()-t0:.0f}s)')

# ── PATH setup ─────────────────────────────────────────────────────────────
# diamond extracts to /content directly (not bin/ subfolder)
for p in ['/content/foldseek/bin', '/content/mmseqs/bin', '/content']:
    if p not in os.environ.get('PATH', ''):
        os.environ['PATH'] = p + ':' + os.environ['PATH']

# ── Verify all tools ──────────────────────────────────────────────────────
print('\nTool versions:')
tool_versions = {}
for tool, cmd in [
    ('foldseek',     'foldseek version'),
    ('mmseqs',       'mmseqs version'),
    ('diamond',      '/content/diamond version'),
    ('blast+',       'blastp -version 2>&1 | head -1'),
    ('hmmer',        'hmmscan -h 2>&1 | head -2 | tail -1'),
    ('amrfinderplus', f'{amrfinder_bin} --version 2>&1 | head -1'),
]:
    v = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip().split('\n')[0]
    tool_versions[tool] = v
    print(f'  {tool:15}: {v}')

# Store for provenance
CONFIG['tool_versions'] = tool_versions
CONFIG['amrfinder_bin'] = amrfinder_bin
CONFIG['amrfinder_db_date'] = amrf_db_date
CONFIG['run_start'] = datetime.now().isoformat()
print('\n✓ Cell 1 complete')


## Cell 2: Parallel Downloads

In [ ]:
# ── CELL 2: Parallel downloads ────────────────────────────────────────────────
# ~8 min — all 5 sources downloaded simultaneously
from concurrent.futures import ThreadPoolExecutor

DATA = CONFIG['local_data_dir']

DOWNLOADS = [
    (CONFIG['card_url'],
     f'{DATA}/card/card.tar.bz2', 'CARD v4.0.1'),
    (CONFIG['afdb_tar_url'],
     f'{DATA}/neisseria/NEIG1_v6.tar', 'NEIG1 AFDB v6'),
    ('https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=proteome:{proteome}'.format(
        proteome=CONFIG['uniprot_proteome']),
     f'{DATA}/neisseria/NEIG1.fasta', 'NEIG1 FASTA'),
    ('https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=proteome:UP000005640+reviewed:true',
     f'{DATA}/human/human_swissprot.fasta', 'Human SwissProt'),
    # UniProt SwissProt KW-0046 bacteria only (taxonomy_id:2)
    (CONFIG['swissprot_amr_url'],
     f'{DATA}/swissprot/amr_bacteria_swissprot.fasta', 'SwissProt AMR bacteria'),
]

def download_file(url, out, label):
    t0 = time.time()
    try:
        subprocess.run(['wget', '-q', url, '-O', out], check=True)
        size = Path(out).stat().st_size / 1e6
        n = 0
        if out.endswith('.fasta'):
            n = int(subprocess.run(f'grep -c "^>" {out}', shell=True,
                                   capture_output=True, text=True).stdout.strip())
        info = f'{size:.1f} MB' + (f' | {n:,} seqs' if n else '')
        print(f'  ✓ {label}: {info} ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f'  ✗ {label}: {e}')

print('Downloading in parallel...')
t0 = time.time()
threads = [threading.Thread(target=download_file, args=(*d,)) for d in DOWNLOADS]
for t in threads: t.start()
for t in threads: t.join()
print(f'✓ Downloads complete in {time.time()-t0:.0f}s')

# ── Validate protein FASTAs (no DNA sequences) ─────────────────────────────
def validate_protein_fasta(path):
    """Check FASTA contains only amino acid sequences, not DNA."""
    dna_only = set('ACGTN')
    aa_valid  = set('ACDEFGHIKLMNPQRSTVWYX*-')
    with open(path) as f:
        for line in f:
            if line.startswith('>'): continue
            seq = set(line.strip().upper())
            if seq and seq.issubset(dna_only):
                return False, f'DNA sequence detected in {path}'
    return True, 'OK'

for fasta, label in [
    (f'{DATA}/swissprot/amr_bacteria_swissprot.fasta', 'SwissProt AMR'),
]:
    ok, msg = validate_protein_fasta(fasta)
    print(f'  {"✓" if ok else "✗"} {label} protein validation: {msg}')

# Record access dates
CONFIG['card_accessed']       = datetime.now().isoformat()
CONFIG['swissprot_accessed']  = datetime.now().isoformat()
CONFIG['afdb_accessed']       = datetime.now().isoformat()

# Count sequences
n_sw = int(subprocess.run(f'grep -c "^>" {DATA}/swissprot/amr_bacteria_swissprot.fasta',
                           shell=True, capture_output=True, text=True).stdout.strip())
CONFIG['swissprot_amr_bacteria_count'] = n_sw
print(f'\nSwissProt AMR bacteria: {n_sw:,} sequences')
print('✓ Cell 2 complete')


## Cell 3: Extract + ProstT5 Weights

In [ ]:
# ── CELL 3: Extract + ProstT5 weights (parallel) ─────────────────────────────
DATA = CONFIG['local_data_dir']

def extract_card():
    t0 = time.time()
    subprocess.run(f'cd {DATA}/card && tar -xjf card.tar.bz2', shell=True)
    n = int(subprocess.run(
        f'grep -c "^>" {DATA}/card/protein_fasta_protein_homolog_model.fasta',
        shell=True, capture_output=True, text=True).stdout.strip())
    print(f'  ✓ CARD: {n:,} homolog model sequences ({time.time()-t0:.0f}s)')
    return n

def extract_neig1():
    t0 = time.time()
    subprocess.run(f'cd {DATA}/neisseria && tar -xf NEIG1_v6.tar', shell=True)
    cif = int(subprocess.run(f'ls {DATA}/neisseria/*.cif.gz | wc -l',
                             shell=True, capture_output=True, text=True).stdout.strip())
    print(f'  ✓ NEIG1: {cif:,} CIF structures ({time.time()-t0:.0f}s)')
    return cif

def get_prostt5():
    t0 = time.time()
    subprocess.run(f'foldseek databases ProstT5 {DATA}/prostt5/weights {CONFIG["local_tmp_dir"]}/prostt5',
                   shell=True, env=os.environ, capture_output=True)
    print(f'  ✓ ProstT5 weights ({time.time()-t0:.0f}s)')

print('Extracting in parallel...')
with ThreadPoolExecutor(max_workers=3) as ex:
    f1 = ex.submit(extract_card)
    f2 = ex.submit(extract_neig1)
    f3 = ex.submit(get_prostt5)

N_CARD  = f1.result()
N_NEIG1 = f2.result()

N_NEIG1_FASTA = int(subprocess.run(f'grep -c "^>" {DATA}/neisseria/NEIG1.fasta',
                                    shell=True, capture_output=True, text=True).stdout.strip())
N_HUMAN = int(subprocess.run(f'grep -c "^>" {DATA}/human/human_swissprot.fasta',
                               shell=True, capture_output=True, text=True).stdout.strip())
N_SWISSPROT = CONFIG['swissprot_amr_bacteria_count']

print(f'\nCounts:')
print(f'  CARD protein homolog: {N_CARD:,}')
print(f'  NEIG1 CIF:            {N_NEIG1:,}')
print(f'  NEIG1 FASTA:          {N_NEIG1_FASTA:,}')
print(f'  Human SwissProt:      {N_HUMAN:,}')
print(f'  SwissProt AMR bact:   {N_SWISSPROT:,}')
print('✓ Cell 3 complete')


## Cell 4: Build Databases + Combined AMR Query

In [ ]:
# ── CELL 4: Build databases + combined AMR protein query ──────────────────────
# ~5 min
import pandas as pd
DATA = CONFIG['local_data_dir']
TMP  = CONFIG['local_tmp_dir']

# ── Check checkpoint ──────────────────────────────────────────────────────
COMBINED_FASTA = f'{DATA}/combined_amr_nr.fasta'
if checkpoint_exists('combined_amr_nr.fasta'):
    load_checkpoint('combined_amr_nr.fasta', COMBINED_FASTA)
    N_COMBINED = int(subprocess.run(f'grep -c "^>" {COMBINED_FASTA}',
                                     shell=True, capture_output=True, text=True).stdout.strip())
    print(f'✓ Combined DB loaded from checkpoint: {N_COMBINED:,} sequences')
else:
    # Tag sequences with source before combining
    # CARD: add prefix |CARD| to header
    # SwissProt: add prefix |SWISSPROT| to header
    card_fasta = f'{DATA}/card/protein_fasta_protein_homolog_model.fasta'
    sw_fasta   = f'{DATA}/swissprot/amr_bacteria_swissprot.fasta'
    tagged_card = f'{DATA}/card/card_tagged.fasta'
    tagged_sw   = f'{DATA}/swissprot/swissprot_tagged.fasta'

    # Tag CARD sequences
    subprocess.run(f"sed 's/^>/>[CARD]/' {card_fasta} > {tagged_card}", shell=True)
    # Tag SwissProt sequences
    subprocess.run(f"sed 's/^>/>[SWISSPROT]/' {sw_fasta} > {tagged_sw}", shell=True)

    # Combine
    subprocess.run(f'cat {tagged_card} {tagged_sw} > {DATA}/combined_amr_raw.fasta', shell=True)
    n_raw = int(subprocess.run(f'grep -c "^>" {DATA}/combined_amr_raw.fasta',
                                shell=True, capture_output=True, text=True).stdout.strip())
    print(f'Combined raw: {n_raw:,} sequences (CARD + SwissProt)')

    # Cluster at 90% NR
    print(f'Clustering at {CONFIG["cluster_identity"]*100:.0f}% identity...')
    t0 = time.time()
    subprocess.run(
        f'mmseqs easy-cluster {DATA}/combined_amr_raw.fasta '
        f'{DATA}/combined_amr_clust {TMP}/cluster '
        f'--min-seq-id {CONFIG["cluster_identity"]} -c 0.8 --threads 4 -v 1',
        shell=True)
    elapsed = time.time() - t0

    # Copy representative sequences to final location
    shutil.copy(f'{DATA}/combined_amr_clust_rep_seq.fasta', COMBINED_FASTA)
    N_COMBINED = int(subprocess.run(f'grep -c "^>" {COMBINED_FASTA}',
                                     shell=True, capture_output=True, text=True).stdout.strip())
    print(f'✓ Combined AMR DB: {N_COMBINED:,} non-redundant sequences ({elapsed:.0f}s)')

    # Save cluster membership TSV
    MEMBERSHIP_TSV = f'{CONFIG["local_results_dir"]}/cluster_membership.tsv'
    shutil.copy(f'{DATA}/combined_amr_clust_cluster.tsv', MEMBERSHIP_TSV)

    # ── Analyse cluster membership ──────────────────────────────────────
    df_clust = pd.read_csv(MEMBERSHIP_TSV, sep='\t', names=['rep','member'])
    # Track source of representative
    df_clust['rep_source']    = df_clust['rep'].apply(
        lambda x: 'CARD' if '[CARD]' in x else 'SWISSPROT')
    df_clust['member_source'] = df_clust['member'].apply(
        lambda x: 'CARD' if '[CARD]' in x else 'SWISSPROT')

    n_card_rep = (df_clust.groupby('rep')['rep_source'].first() == 'CARD').sum()
    n_sw_rep   = (df_clust.groupby('rep')['rep_source'].first() == 'SWISSPROT').sum()
    n_sw_in_card_cluster = df_clust[
        (df_clust['rep_source']=='CARD') & (df_clust['member_source']=='SWISSPROT')
    ].shape[0]
    n_sw_unique = n_sw_rep

    print(f'\nCluster membership:')
    print(f'  Representatives from CARD:       {n_card_rep:,}')
    print(f'  Representatives from SwissProt:  {n_sw_rep:,} (novel clusters)')
    print(f'  SwissProt seqs in CARD clusters: {n_sw_in_card_cluster:,} (overlap)')

    CONFIG['cluster_stats'] = {
        'n_card_reps': int(n_card_rep),
        'n_swissprot_reps': int(n_sw_rep),
        'n_swissprot_in_card_clusters': int(n_sw_in_card_cluster),
    }

    # Save checkpoints
    save_checkpoint(COMBINED_FASTA, 'combined_amr_nr.fasta')
    save_checkpoint(MEMBERSHIP_TSV, 'cluster_membership.tsv')

QUERY_FASTA = COMBINED_FASTA

# ── Build Foldseek NEIG1 structure DB ─────────────────────────────────────
!foldseek createdb {DATA}/neisseria/ {DATA}/dbs/neig1_db --file-include 'cif.gz$' -v 1
N_DB = int(open(f'{DATA}/dbs/neig1_db.lookup').read().count('\n'))
assert 2000 <= N_DB <= 2200, f'Unexpected NEIG1 DB size: {N_DB}'
print(f'✓ NEIG1 structure DB: {N_DB:,}')

# ── Build DIAMOND DB ─────────────────────────────────────────────────────
!/content/diamond makedb --in {DATA}/neisseria/NEIG1.fasta \
    --db {DATA}/dbs/neig1_diamond --quiet
print('✓ DIAMOND DB built')

# ── Preprocess NEIG1 FASTA for AMRFinderPlus (strip pipe chars in headers) ──
!sed 's/>sp|\([^|]*\)|.*/\>\1/' {DATA}/neisseria/NEIG1.fasta \
    > {DATA}/neisseria/NEIG1_amrfinder.fasta
n_amrf = int(subprocess.run(f'grep -c "^>" {DATA}/neisseria/NEIG1_amrfinder.fasta',
                              shell=True, capture_output=True, text=True).stdout.strip())
print(f'✓ AMRFinderPlus FASTA: {n_amrf:,} sequences (headers cleaned)')
print('\n✓ Cell 4 complete')


## Cell 5a: Foldseek Structural Search (GPU, ~55 min)

In [ ]:
# ── CELL 5a: Foldseek search — GPU alone, maximum sensitivity ─────────────────
# ~55 min on T4 with s=9.5
# IMPORTANT: Runs ALONE — do not parallelize with other GPU/CPU tasks
# Checkpoint saves to Drive after completion

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']
TMP     = CONFIG['local_tmp_dir']
FS_OUT  = f'{RESULTS}/foldseek_raw.tsv'

E_THRESH       = CONFIG['evalue_threshold']
QCOV_MIN       = CONFIG['qcov_min']
TCOV_MIN       = CONFIG['tcov_min']
PIDENT_CRYPTIC = CONFIG['pident_cryptic']

if checkpoint_exists('foldseek_raw.tsv'):
    load_checkpoint('foldseek_raw.tsv', FS_OUT)
    n_raw = int(subprocess.run(f'wc -l < {FS_OUT}',
                                shell=True, capture_output=True, text=True).stdout.strip())
    print(f'✓ Foldseek loaded from checkpoint: {n_raw:,} raw hits')
else:
    # Clean tmp before run
    subprocess.run(f'rm -rf {TMP}/foldseek/* && mkdir -p {TMP}/foldseek', shell=True)

    print(f'Starting Foldseek+ProstT5 (s={CONFIG["foldseek_sensitivity"]})...')
    print('Expected: ~55 min on T4')
    t0 = time.time()

    r = subprocess.run(
        f'foldseek easy-search {QUERY_FASTA} {DATA}/dbs/neig1_db '
        f'{FS_OUT} {TMP}/foldseek '
        f'--prostt5-model {DATA}/prostt5/weights --gpu 1 '
        f'-e 0.001 --threads 4 '
        f'-s {CONFIG["foldseek_sensitivity"]} '
        f'--format-output query,target,pident,evalue,qlen,tlen,alnlen,bits,tstart,tend,qstart,qend',
        shell=True, env=os.environ, capture_output=True
    )

    elapsed = time.time() - t0
    n_raw = int(subprocess.run(f'wc -l < {FS_OUT}',
                                shell=True, capture_output=True, text=True).stdout.strip()) \
            if Path(FS_OUT).exists() else 0

    if r.returncode != 0:
        print(f'✗ Foldseek FAILED after {elapsed:.0f}s')
        print(r.stderr.decode()[:500])
    else:
        print(f'✓ Foldseek: {n_raw:,} raw hits in {elapsed:.0f}s')
        CONFIG['foldseek_runtime_s'] = round(elapsed)

        # Sanity check
        assert n_raw > 1000, f'Foldseek raw hits too low ({n_raw}) — ProstT5/GPU may have failed'
        print(f'✓ Sanity check passed')

        # Save checkpoint immediately
        save_checkpoint(FS_OUT, 'foldseek_raw.tsv')

print('\n✓ Cell 5a complete — safe to start Session 2 from here if needed')


## Cell 5b: CPU Searches (MMseqs2 + DIAMOND + Human Screen)

In [ ]:
# ── CELL 5b: CPU searches — parallel, separate tmp dirs ───────────────────────
# ~5 min — uses PROTEIN query only for all sequence methods

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']
TMP     = CONFIG['local_tmp_dir']

# Restore if session restart
if 'QUERY_FASTA' not in dir():
    QUERY_FASTA = f'{DATA}/combined_amr_nr.fasta'
    load_checkpoint('combined_amr_nr.fasta', QUERY_FASTA)

MM_OUT  = f'{RESULTS}/mmseqs_raw.tsv'
DI_OUT  = f'{RESULTS}/diamond_raw.tsv'
HU_OUT  = f'{RESULTS}/neig1_vs_human.tsv'

# Check checkpoints
mm_exists = checkpoint_exists('mmseqs_raw.tsv')
di_exists = checkpoint_exists('diamond_raw.tsv')
hu_exists = checkpoint_exists('human_raw.tsv')

if mm_exists: load_checkpoint('mmseqs_raw.tsv', MM_OUT)
if di_exists: load_checkpoint('diamond_raw.tsv', DI_OUT)
if hu_exists: load_checkpoint('human_raw.tsv',   HU_OUT)

# Only run missing searches
searches_to_run = []
if not mm_exists:
    searches_to_run.append((
        f'mmseqs easy-search {QUERY_FASTA} {DATA}/neisseria/NEIG1.fasta '
        f'{MM_OUT} {TMP}/mmseqs '
        f'--format-output query,target,pident,evalue,qlen,tlen,alnlen,bits,tstart,tend,qstart,qend '
        f'-e 0.001 --threads 4 -s {CONFIG["foldseek_sensitivity"]}',
        'MMseqs2', MM_OUT))
if not di_exists:
    os.makedirs('/tmp/diamond_search', exist_ok=True)
    searches_to_run.append((
        f'/content/diamond blastp --query {QUERY_FASTA} --db {DATA}/dbs/neig1_diamond '
        f'--out {DI_OUT} '
        f'--outfmt 6 qseqid sseqid pident evalue qlen slen length bitscore qstart qend sstart send '
        f'--evalue 0.001 --threads 4 --sensitive '
        f'--tmpdir /tmp/diamond_search',
        'DIAMOND', DI_OUT))
if not hu_exists:
    searches_to_run.append((
        f'mmseqs easy-search {DATA}/neisseria/NEIG1.fasta '
        f'{DATA}/human/human_swissprot.fasta '
        f'{HU_OUT} {TMP}/human '
        f'--format-output query,target,pident,evalue,qlen,tlen,alnlen '
        f'-e 1e-5 --threads 4 -s 7.5',
        'Human screen', HU_OUT))

def run_cpu_search(cmd, label, out):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, env=os.environ, capture_output=True)
    elapsed = time.time() - t0
    n = int(subprocess.run(f'wc -l < {out}', shell=True,
                            capture_output=True, text=True).stdout.strip()) \
        if Path(out).exists() else 0
    status = '✓' if r.returncode == 0 else '✗'
    print(f'  {status} {label}: {n:,} raw hits in {elapsed:.0f}s')
    if r.returncode != 0:
        print(f'    ERROR: {r.stderr.decode()[:400]}')
    return r.returncode == 0, elapsed

if searches_to_run:
    print(f'Running {len(searches_to_run)} searches in parallel...')
    t0 = time.time()
    threads = [threading.Thread(target=run_cpu_search, args=s) for s in searches_to_run]
    for t in threads: t.start()
    for t in threads: t.join()
    print(f'✓ CPU searches complete in {time.time()-t0:.0f}s')

    # Save checkpoints
    if not mm_exists: save_checkpoint(MM_OUT, 'mmseqs_raw.tsv')
    if not di_exists: save_checkpoint(DI_OUT, 'diamond_raw.tsv')
    if not hu_exists: save_checkpoint(HU_OUT, 'human_raw.tsv')
else:
    print('✓ All CPU searches loaded from checkpoints')

# Sanity checks
mm_n = int(subprocess.run(f'wc -l < {MM_OUT}', shell=True, capture_output=True, text=True).stdout.strip())
di_n = int(subprocess.run(f'wc -l < {DI_OUT}', shell=True, capture_output=True, text=True).stdout.strip())
assert mm_n > 50, f'MMseqs2 raw hits too low ({mm_n})'
assert di_n > 50, f'DIAMOND raw hits too low ({di_n}) — check /content/diamond path'
print(f'✓ Sanity: MMseqs2={mm_n:,} | DIAMOND={di_n:,}')
print('\n✓ Cell 5b complete')


## Cell 5c: AMRFinderPlus HMM Search (Independent)

In [ ]:
# ── CELL 5c: AMRFinderPlus — HMM-based, fully independent ────────────────────
# ~8 min — uses NEIG1 protein FASTA, NOT query DB
# Runs without --organism flag → consistent with "acquired genes" scope
# Reports both BLAST+HMM hits and PARTIAL hits separately

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']
AMRF_OUT = f'{RESULTS}/amrfinder_out.tsv'

if checkpoint_exists('amrfinder_out.tsv'):
    load_checkpoint('amrfinder_out.tsv', AMRF_OUT)
    print('✓ AMRFinderPlus loaded from checkpoint')
else:
    amrfinder_bin = CONFIG['amrfinder_bin']
    amrf_db = f'{DATA}/amrfinder/db'

    print('Running AMRFinderPlus (HMM + BLAST)...')
    t0 = time.time()
    r = subprocess.run(
        f'{amrfinder_bin} '
        f'--protein {DATA}/neisseria/NEIG1_amrfinder.fasta '
        f'--database {amrf_db} '
        f'--output {AMRF_OUT} '
        f'--threads 4 '
        f'--plus',    # include stress/virulence — we filter to AMR later
        shell=True, capture_output=True, text=True
    )
    elapsed = time.time() - t0

    if r.returncode != 0:
        print(f'✗ AMRFinderPlus FAILED: {r.stderr[:300]}')
    else:
        print(f'✓ AMRFinderPlus complete in {elapsed:.0f}s')
        CONFIG['amrfinder_runtime_s'] = round(elapsed)
        save_checkpoint(AMRF_OUT, 'amrfinder_out.tsv')

# ── Parse AMRFinderPlus output ─────────────────────────────────────────────
import pandas as pd

df_amrf_raw = pd.read_csv(AMRF_OUT, sep='\t')
print(f'\nAMRFinderPlus raw output: {len(df_amrf_raw)} entries')
print(f'Columns: {list(df_amrf_raw.columns)}')

# Filter: AMR element type only (exclude STRESS, VIRULENCE)
df_amrf_amr = df_amrf_raw[df_amrf_raw['Element type'] == 'AMR'].copy()
print(f'AMR elements only: {len(df_amrf_amr)}')

# Separate by method
df_amrf_full    = df_amrf_amr[df_amrf_amr['Method'].isin(['BLAST', 'HMM'])].copy()
df_amrf_partial = df_amrf_amr[df_amrf_amr['Method'] == 'PARTIAL'].copy()

print(f'\nAMRFinderPlus breakdown:')
print(f'  BLAST+HMM (full hits): {len(df_amrf_full)}')
print(f'  PARTIAL hits:          {len(df_amrf_partial)}')
print(f'  Method distribution:')
print(df_amrf_amr['Method'].value_counts().to_string())

# Map IDs back to UniProt accessions
amrfinder_ids_full    = set(df_amrf_full['Protein identifier'].unique())
amrfinder_ids_partial = set(df_amrf_partial['Protein identifier'].unique())
amrfinder_ids_all     = amrfinder_ids_full | amrfinder_ids_partial

print(f'\nUnique NEIG1 proteins:')
print(f'  BLAST+HMM: {len(amrfinder_ids_full)}')
print(f'  PARTIAL:   {len(amrfinder_ids_partial)}')
print(f'  All AMR:   {len(amrfinder_ids_all)}')
print('\n✓ Cell 5c complete')


## Cell 6: ESM2 — Statistically Calibrated Threshold

In [ ]:
# ── CELL 6: ESM2 pLM — null distribution calibration + search ─────────────────
# ~25 min — statistically calibrated threshold via NEIG1-NEIG1 null distribution

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from scipy import stats
import numpy as np

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']

# Restore variables if session restart
if 'QUERY_FASTA' not in dir():
    QUERY_FASTA = f'{DATA}/combined_amr_nr.fasta'

print(f'Loading {CONFIG["esm2_model"]}...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['esm2_model'])
esm2 = AutoModel.from_pretrained(CONFIG['esm2_model']).cuda().eval()
print('✓ ESM2 loaded')

def load_fasta(path, id_parse='uniprot'):
    seqs, cur_id, cur_seq = {}, '', ''
    with open(path) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: seqs[cur_id] = cur_seq
                h = line[1:].strip()
                cur_id = h.split('|')[1] if (id_parse == 'uniprot' and '|' in h) else h.split()[0]
                cur_seq = ''
            else:
                cur_seq += line.strip()
    if cur_id: seqs[cur_id] = cur_seq
    return seqs

def embed_batch(seqs_list, batch=32, maxlen=512):
    all_embs = []
    for i in range(0, len(seqs_list), batch):
        b = [s[:maxlen] for s in seqs_list[i:i+batch]]
        inp = tokenizer(b, return_tensors='pt', padding=True,
                        truncation=True, max_length=maxlen).to('cuda')
        with torch.no_grad():
            out = esm2(**inp).last_hidden_state.mean(1).cpu()
        all_embs.append(out)
    return torch.cat(all_embs)

# Embed NEIG1 proteins
neig1_seqs = load_fasta(f'{DATA}/neisseria/NEIG1.fasta')
neig1_ids  = list(neig1_seqs.keys())
print(f'Embedding {len(neig1_seqs):,} NEIG1 proteins...')
t0 = time.time()
neig1_embs = embed_batch(list(neig1_seqs.values()))
print(f'✓ NEIG1 embeddings: {neig1_embs.shape} ({time.time()-t0:.0f}s)')

# ── Step 1: Null distribution (NEIG1 vs NEIG1 random pairs) ───────────────
print(f'\nComputing null distribution ({CONFIG["esm2_null_pairs"]:,} random pairs)...')
rng = np.random.RandomState(SEED)
n_neig1 = len(neig1_ids)
idx1 = rng.choice(n_neig1, CONFIG['esm2_null_pairs'] * 2)
idx2 = rng.choice(n_neig1, CONFIG['esm2_null_pairs'] * 2)
mask = idx1 != idx2  # no self-comparisons
idx1, idx2 = idx1[mask][:CONFIG['esm2_null_pairs']], idx2[mask][:CONFIG['esm2_null_pairs']]

null_sims = F.cosine_similarity(
    neig1_embs[idx1],
    neig1_embs[idx2],
    dim=-1
).numpy()

# ── Step 2: Fit distribution ──────────────────────────────────────────────
# Test normality
stat, pval_norm = stats.shapiro(null_sims[:5000])  # Shapiro-Wilk on subset
print(f'Shapiro-Wilk normality test: stat={stat:.4f}, p={pval_norm:.4e}')

if pval_norm > 0.05:
    # Normal distribution
    mu, sigma = null_sims.mean(), null_sims.std()
    dist_name = 'normal'
    SIM_THRESHOLD = stats.norm.ppf(1 - CONFIG['esm2_pvalue'], loc=mu, scale=sigma)
else:
    # Fit beta distribution (bounded 0-1)
    # Shift and scale cosine sims to [0,1] for beta fitting
    a, b, loc, scale = stats.beta.fit(null_sims, floc=null_sims.min()-0.001,
                                       fscale=null_sims.max()-null_sims.min()+0.002)
    dist_name = 'beta'
    SIM_THRESHOLD = stats.beta.ppf(1 - CONFIG['esm2_pvalue'], a, b, loc=loc, scale=scale)

print(f'\nNull distribution ({dist_name}):')
print(f'  Mean:   {null_sims.mean():.4f}')
print(f'  Std:    {null_sims.std():.4f}')
print(f'  Min:    {null_sims.min():.4f}')
print(f'  Max:    {null_sims.max():.4f}')
print(f'  Threshold at p={CONFIG["esm2_pvalue"]:.0e}: {SIM_THRESHOLD:.4f}')

CONFIG['esm2_null_dist']      = dist_name
CONFIG['esm2_null_mean']      = float(null_sims.mean())
CONFIG['esm2_null_std']       = float(null_sims.std())
CONFIG['esm2_threshold']      = float(SIM_THRESHOLD)

# Save null sims for figure
np.save(f'{RESULTS}/esm2_null_sims.npy', null_sims)

# ── Step 3: Check checkpoint for ESM2 hits ────────────────────────────────
ESM2_HITS_PATH = f'{RESULTS}/esm2_hits.json'
if checkpoint_exists('esm2_hits.json'):
    load_checkpoint('esm2_hits.json', ESM2_HITS_PATH)
    with open(ESM2_HITS_PATH) as f:
        esm2_hits = json.load(f)
    print(f'✓ ESM2 hits loaded from checkpoint: {len(esm2_hits):,}')
else:
    # ── Step 4: AMR query search ──────────────────────────────────────────
    amr_seqs  = load_fasta(QUERY_FASTA, id_parse='first')
    amr_list  = list(amr_seqs.items())
    esm2_hits = {}

    print(f'\nSearching {len(amr_list):,} AMR queries vs NEIG1 (threshold={SIM_THRESHOLD:.4f})...')
    t0 = time.time()
    for i in range(0, len(amr_list), 100):
        batch_ids  = [x[0] for x in amr_list[i:i+100]]
        embs = embed_batch([x[1] for x in amr_list[i:i+100]], batch=50)
        sims = F.cosine_similarity(embs.unsqueeze(1), neig1_embs.unsqueeze(0), dim=-1)
        for j, aid in enumerate(batch_ids):
            max_sim, max_idx = sims[j].max(0)
            if max_sim.item() >= SIM_THRESHOLD:
                nid = neig1_ids[max_idx.item()]
                if nid not in esm2_hits or max_sim.item() > esm2_hits[nid][1]:
                    esm2_hits[nid] = [aid, float(max_sim.item())]
        if i % 500 == 0:
            print(f'  {i:,}/{len(amr_list):,} | hits: {len(esm2_hits):,}')

    print(f'✓ ESM2: {len(esm2_hits):,} hits in {time.time()-t0:.0f}s')
    with open(ESM2_HITS_PATH, 'w') as f:
        json.dump(esm2_hits, f)
    save_checkpoint(ESM2_HITS_PATH, 'esm2_hits.json')

esm2_ids = set(esm2_hits.keys())
print(f'\n✓ ESM2 unique NEIG1 hits: {len(esm2_ids):,}')
print(f'  Threshold used: {SIM_THRESHOLD:.4f} (p={CONFIG["esm2_pvalue"]:.0e})')
print('\n✓ Cell 6 complete')


## Cell 7: Parse All Results + Uniform Filters + Sanity Checks

In [ ]:
# ── CELL 7: Parse results + uniform filters + sanity checks ───────────────────
import pandas as pd

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']

E_THRESH       = CONFIG['evalue_threshold']
QCOV_MIN       = CONFIG['qcov_min']
TCOV_MIN       = CONFIG['tcov_min']
PIDENT_CRYPTIC = CONFIG['pident_cryptic']

# Restore variables if session restart
if 'esm2_ids' not in dir():
    with open(f'{RESULTS}/esm2_hits.json') as f:
        esm2_hits = json.load(f)
    esm2_ids = set(esm2_hits.keys())

def parse_tsv(path, id_type='af', has_positions=True):
    if has_positions:
        cols = ['query','target','pident','evalue','qlen','tlen','alnlen',
                'bits','tstart','tend','qstart','qend']
    else:
        cols = ['query','target','pident','evalue','qlen','tlen','alnlen']
    df = pd.read_csv(path, sep='\t', names=cols[:sum(1 for _ in open(path).readline().split('\t'))])
    # Coverage — clip at 1.0 (Foldseek circular permutation artifact)
    df['qcov'] = (df['alnlen'] / df['qlen']).clip(upper=1.0)
    df['tcov'] = (df['alnlen'] / df['tlen']).clip(upper=1.0)
    if id_type == 'af':
        df['neig1_id'] = df['target'].apply(
            lambda x: x.split('-')[1] if x.startswith('AF-') else x)
    else:
        df['neig1_id'] = df['target'].apply(
            lambda x: x.split('|')[1] if '|' in x else x.split()[0])
    # Track source of query (CARD vs SWISSPROT)
    df['query_source'] = df['query'].apply(
        lambda x: 'CARD' if '[CARD]' in x else ('SWISSPROT' if '[SWISSPROT]' in x else 'CARD'))
    return df

def strict_best(df):
    filt = df[(df['evalue'] < E_THRESH) &
              (df['qcov']   >= QCOV_MIN) &
              (df['tcov']   >= TCOV_MIN)]
    return filt.sort_values('evalue').groupby('neig1_id').first().reset_index()

def strict_all(df):
    """All hits passing filters — for multi-domain analysis."""
    return df[(df['evalue'] < E_THRESH) &
              (df['qcov']   >= QCOV_MIN) &
              (df['tcov']   >= TCOV_MIN)].copy()

# Parse all methods
print('Parsing search results...')
df_fs  = parse_tsv(f'{RESULTS}/foldseek_raw.tsv',        'af',  True)
df_mm  = parse_tsv(f'{RESULTS}/mmseqs_raw.tsv',          'seq', True)
df_di  = parse_tsv(f'{RESULTS}/diamond_raw.tsv',         'seq', True)
df_hu  = parse_tsv(f'{RESULTS}/neig1_vs_human.tsv',      'seq', False)

df_fs_best = strict_best(df_fs)
df_mm_best = strict_best(df_mm)
df_di_best = strict_best(df_di)
df_fs_all  = strict_all(df_fs)   # for multi-domain analysis

# Human screen
df_hu['neig1_id'] = df_hu['query'].apply(
    lambda x: x.split('|')[1] if '|' in x else x)
has_human = set(df_hu[df_hu['neig1_id'].isin(df_fs_best['neig1_id'])]['neig1_id'])
no_human  = set(df_fs_best['neig1_id']) - has_human

fold_ids    = set(df_fs_best['neig1_id'])
mmseqs_ids  = set(df_mm_best['neig1_id'])
diamond_ids = set(df_di_best['neig1_id'])

# Coverage >1.0 cases
n_cov_gt1 = (df_fs['qcov'] > 1.0).sum()

# ── Sanity checks ──────────────────────────────────────────────────────────
print('\nSanity checks:')
assert len(fold_ids) > len(mmseqs_ids),  f'FAIL: Foldseek ({len(fold_ids)}) <= MMseqs2 ({len(mmseqs_ids)})'
assert len(fold_ids) > len(diamond_ids), f'FAIL: Foldseek ({len(fold_ids)}) <= DIAMOND ({len(diamond_ids)})'
assert len(diamond_ids) > 0,             f'FAIL: DIAMOND = 0 hits'
assert len(fold_ids) > 100,              f'FAIL: Foldseek hits too low ({len(fold_ids)})'
print(f'  ✓ Foldseek > MMseqs2 ({len(fold_ids)} > {len(mmseqs_ids)})')
print(f'  ✓ Foldseek > DIAMOND ({len(fold_ids)} > {len(diamond_ids)})')
print(f'  ✓ DIAMOND > 0')
print(f'  ✓ Foldseek > 100 hits')

print(f'\nStrict hits (e<{E_THRESH}, qcov>={QCOV_MIN}, tcov>={TCOV_MIN}):')
print(f'  Foldseek:       {len(fold_ids):>5,}')
print(f'  MMseqs2:        {len(mmseqs_ids):>5,}')
print(f'  DIAMOND:        {len(diamond_ids):>5,}')
print(f'  ESM2:           {len(esm2_ids):>5,} (threshold={CONFIG.get("esm2_threshold",0):.4f}, p={CONFIG["esm2_pvalue"]:.0e})')
print(f'  AMRFinder full: {len(amrfinder_ids_full):>5,}')
print(f'  AMRFinder all:  {len(amrfinder_ids_all):>5,}')
print(f'  Priority targets (no human homolog): {len(no_human):,}')
print(f'  Coverage >1.0 capped: {n_cov_gt1} (Foldseek circular permutation)')
print('\n✓ Cell 7 complete')


## Cell 8: Hybrid Annotation (CARD + SwissProt)

In [ ]:
# ── CELL 8: Hybrid annotation — CARD ARO if CARD rep, SwissProt if SwissProt rep ──
import pandas as pd

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']

# ── Load CARD ARO metadata ─────────────────────────────────────────────────
aro_lookup, aro_meta = {}, {}
card_fasta = f'{DATA}/card/protein_fasta_protein_homolog_model.fasta'
with open(card_fasta) as f:
    for line in f:
        if not line.startswith('>'): continue
        parts = line[1:].strip().split('|')
        # Header after [CARD] tag: [CARD]gb|ACC|ARO:XXX|...
        clean = line[1:].replace('[CARD]', '').strip()
        parts = clean.split('|')
        if len(parts) >= 3 and parts[2].startswith('ARO:'):
            aro_lookup[parts[1].strip()] = parts[2].strip()

with open(f'{DATA}/card/aro_index.tsv') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        aro_meta[row['ARO Accession'].strip()] = {
            'gene_family': row.get('AMR Gene Family', ''),
            'drug_class':  row.get('Drug Class', ''),
            'mechanism':   row.get('Resistance Mechanism', ''),
            'short_name':  row.get('CARD Short Name', '')
        }

# ── Load SwissProt metadata from FASTA headers ─────────────────────────────
# SwissProt header: [SWISSPROT]>sp|ACC|NAME_ORGANISM Description
sw_meta = {}
with open(f'{DATA}/swissprot/amr_bacteria_swissprot.fasta') as f:
    for line in f:
        if not line.startswith('>'): continue
        h = line[1:].replace('[SWISSPROT]', '').strip()
        parts = h.split('|')
        if len(parts) >= 3:
            acc  = parts[1]
            rest = parts[2] if len(parts) > 2 else ''
            name = rest.split()[0] if rest else ''
            desc = ' '.join(rest.split()[1:]).split(' OS=')[0] if rest else ''
            sw_meta[acc] = {
                'gene_family': desc[:60],
                'drug_class':  '',   # not in FASTA header
                'mechanism':   '',
                'short_name':  name
            }

def annotate_hybrid(df):
    df = df.copy()
    # Determine query source
    df['query_source'] = df['query'].apply(
        lambda x: 'CARD' if '[CARD]' in x else 'SWISSPROT')

    # Clean query ID
    df['query_clean'] = df['query'].apply(
        lambda x: x.replace('[CARD]', '').replace('[SWISSPROT]', '').strip())

    # CARD annotation path
    df['aro'] = df.apply(
        lambda r: aro_lookup.get(r['query_clean'].split('|')[1] if '|' in r['query_clean'] else r['query_clean'], None)
        if r['query_source'] == 'CARD' else None, axis=1)

    for field in ['gene_family', 'drug_class', 'mechanism', 'short_name']:
        # CARD path
        df[field] = df.apply(lambda r:
            aro_meta.get(r['aro'], {}).get(field, '') if r['query_source'] == 'CARD' and pd.notna(r.get('aro')) else '',
            axis=1)
        # SwissProt path (fill where CARD gave nothing)
        mask = df[field].eq('') & (df['query_source'] == 'SWISSPROT')
        df.loc[mask, field] = df.loc[mask, 'query_clean'].apply(
            lambda x: sw_meta.get(x.split('|')[1] if '|' in x else x, {}).get(field, ''))

    df['cryptic']   = df['pident'] < PIDENT_CRYPTIC
    df['has_human'] = df['neig1_id'].isin(has_human)

    # Coverage tier
    df['tcov_tier'] = pd.cut(df['tcov'],
        bins=[0, 0.3, 0.5, 0.7, 0.9, 1.0],
        labels=['<30%', '30-50%', '50-70%', '70-90%', '90-100%'])

    # Identity tier
    df['pident_tier'] = pd.cut(df['pident'],
        bins=[0, 30, 50, 70, 90, 100],
        labels=['<30% cryptic', '30-50% remote', '50-70% moderate', '70-90% close', '>90% near-identical'])

    return df

df_fs_best = annotate_hybrid(df_fs_best)
df_fs_all  = annotate_hybrid(df_fs_all)

n_card_hits = (df_fs_best['query_source'] == 'CARD').sum()
n_sw_hits   = (df_fs_best['query_source'] == 'SWISSPROT').sum()
print(f'Annotation sources:')
print(f'  CARD-derived hits:       {n_card_hits}')
print(f'  SwissProt-derived hits:  {n_sw_hits}')
print(f'  Cryptic (<30% id):       {df_fs_best["cryptic"].sum()} ({df_fs_best["cryptic"].mean()*100:.1f}%)')
print(f'\nTop gene families:')
print(df_fs_best['gene_family'].value_counts().head(10).to_string())
print('\n✓ Cell 8 complete')


## Cell 9: Multi-domain AMR Analysis

In [ ]:
# ── CELL 9: Multi-domain analysis — position-aware non-overlapping hits ──────────
# Identifies NEIG1 proteins with evidence of multiple distinct AMR domains

import pandas as pd
from collections import defaultdict

def find_nonoverlapping_hits(group, min_gap=30):
    """
    For a single NEIG1 protein, find non-overlapping AMR hits on the target.
    Uses target coordinates (tstart, tend) to detect distinct domains.
    min_gap: minimum gap in residues between domains.
    """
    if 'tstart' not in group.columns:
        return group.head(1)  # fallback if no positions

    # Sort by target start position
    hits = group.sort_values('tstart').copy()
    selected = []
    last_end = -min_gap

    for _, row in hits.iterrows():
        tstart = row.get('tstart', 0)
        tend   = row.get('tend', row['alnlen'])
        # Non-overlapping: starts after last selected region + gap
        if tstart >= last_end + min_gap:
            selected.append(row)
            last_end = tend

    return pd.DataFrame(selected) if selected else hits.head(1)

# Apply to all Foldseek hits
print('Identifying multi-domain AMR proteins...')
df_fs_all_sorted = df_fs_all.copy()

# Group by NEIG1 protein, find non-overlapping hits
multi_domain_results = []
for neig1_id, group in df_fs_all_sorted.groupby('neig1_id'):
    nonoverlap = find_nonoverlapping_hits(group)
    for _, row in nonoverlap.iterrows():
        row_dict = row.to_dict()
        row_dict['neig1_id'] = neig1_id
        multi_domain_results.append(row_dict)

df_multidomain = pd.DataFrame(multi_domain_results)

# Count domains per protein
domain_counts = df_multidomain.groupby('neig1_id').size()
n_single  = (domain_counts == 1).sum()
n_double  = (domain_counts == 2).sum()
n_triple  = (domain_counts >= 3).sum()
multi_domain_proteins = domain_counts[domain_counts > 1].index.tolist()

print(f'\nDomain analysis ({len(domain_counts)} NEIG1 proteins with AMR hits):')
print(f'  Single AMR domain:     {n_single} proteins')
print(f'  Two AMR domains:       {n_double} proteins')
print(f'  Three+ AMR domains:    {n_triple} proteins')

if multi_domain_proteins:
    print(f'\nMulti-domain proteins (top 10):')
    for uid in multi_domain_proteins[:10]:
        hits = df_multidomain[df_multidomain['neig1_id'] == uid]
        families = ' | '.join(hits['gene_family'].str[:30].tolist())
        print(f'  {uid}: {len(hits)} domains — {families}')

# Save multi-domain results
df_multidomain.to_csv(f'{RESULTS}/neig1_amr_all_domains.tsv', sep='\t', index=False)

CONFIG['multidomain_stats'] = {
    'single_domain':  int(n_single),
    'double_domain':  int(n_double),
    'triple_plus':    int(n_triple),
    'multi_domain_proteins': multi_domain_proteins[:20]
}
print('\n✓ Cell 9 complete')


## Cell 10: UniProt Parallel Annotation

In [ ]:
# ── CELL 10: UniProt parallel annotation — retry + backoff ───────────────────
# 10-12 workers, 0.1s sleep, 3 retries with exponential backoff

import requests, time, json
from concurrent.futures import ThreadPoolExecutor, as_completed

RESULTS = CONFIG['local_results_dir']
CACHE_PATH = f'{RESULTS}/uniprot_cache.json'

AMR_KW = {
    'Antibiotic resistance', 'Drug resistance', 'Aminoglycoside resistance',
    'Beta-lactam resistance', 'Fluoroquinolone resistance', 'Vancomycin resistance',
    'Tetracycline resistance', 'Macrolide resistance', 'Multidrug resistance', 'Efflux'
}

def fetch_uniprot(uid, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(f'https://rest.uniprot.org/uniprotkb/{uid}.json',
                             timeout=20)
            if r.status_code == 429:
                # Rate limited — exponential backoff + jitter
                wait = (2 ** attempt) + random.uniform(0, 1)
                time.sleep(wait)
                continue
            if r.status_code != 200:
                return uid, {}
            d = r.json()
            genes    = d.get('genes', [])
            gene_nm  = genes[0].get('geneName', {}).get('value', '') if genes else ''
            kws      = [k['name'] for k in d.get('keywords', [])]
            amr_kws  = [k for k in kws if k in AMR_KW]
            go_refs  = [x for x in d.get('dbReferences', []) if x['type'] == 'GO']
            kegg     = [x['id'] for x in d.get('dbReferences', []) if x['type'] == 'KEGG']
            pdb_refs = [x['id'] for x in d.get('dbReferences', []) if x['type'] == 'PDB']
            func     = [c['texts'][0]['value'] for c in d.get('comments', [])
                        if c['commentType'] == 'FUNCTION' and c.get('texts')]
            pname    = (d.get('proteinDescription', {})
                         .get('recommendedName', {}).get('fullName', {}).get('value', ''))
            time.sleep(0.1)  # polite rate limiting
            return uid, {
                'gene_name':    gene_nm,
                'protein_name': pname,
                'amr_keywords': amr_kws,
                'has_amr_kw':   len(amr_kws) > 0,
                'kegg':         kegg,
                'pdb_refs':     pdb_refs,
                'go':           [(x['id'], x.get('properties', {}).get('term', '')) for x in go_refs],
                'function':     func[:1],
            }
        except Exception as e:
            if attempt == max_retries - 1:
                return uid, {'error': str(e)}
            time.sleep(2 ** attempt)
    return uid, {}

# Check checkpoint
if checkpoint_exists('uniprot_cache.json'):
    load_checkpoint('uniprot_cache.json', CACHE_PATH)
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f'✓ UniProt cache loaded: {len(cache):,} entries')
else:
    ids_to_fetch = df_fs_best['neig1_id'].tolist()
    cache = {}
    print(f'Fetching {len(ids_to_fetch):,} UniProt entries (12 workers, 0.1s sleep)...')
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = {ex.submit(fetch_uniprot, uid): uid for uid in ids_to_fetch}
        for i, fut in enumerate(as_completed(futs)):
            uid, res = fut.result()
            cache[uid] = res
            if (i+1) % 50 == 0:
                print(f'  {i+1:,}/{len(ids_to_fetch):,}')
    print(f'✓ Fetched in {time.time()-t0:.0f}s')
    with open(CACHE_PATH, 'w') as f:
        json.dump(cache, f)
    save_checkpoint(CACHE_PATH, 'uniprot_cache.json')

def gf(uid, field, default=''):
    v = cache.get(uid, {}).get(field, default)
    return v if v is not None else default

df_fs_best['uniprot_gene']  = df_fs_best['neig1_id'].apply(lambda x: gf(x, 'gene_name'))
df_fs_best['uniprot_name']  = df_fs_best['neig1_id'].apply(lambda x: gf(x, 'protein_name'))
df_fs_best['amr_keywords']  = df_fs_best['neig1_id'].apply(
    lambda x: ';'.join(gf(x, 'amr_keywords', [])))
df_fs_best['has_amr_kw']    = df_fs_best['neig1_id'].apply(lambda x: gf(x, 'has_amr_kw', False))
df_fs_best['kegg_id']       = df_fs_best['neig1_id'].apply(
    lambda x: gf(x, 'kegg', [''])[0] if gf(x, 'kegg', []) else '')
df_fs_best['pdb_refs']      = df_fs_best['neig1_id'].apply(
    lambda x: ';'.join(gf(x, 'pdb_refs', [])))

# UniProt KW-0046 cross-validation
r_kw = requests.get('https://rest.uniprot.org/uniprotkb/search',
    params={'query': f'proteome:{CONFIG["uniprot_proteome"]} AND keyword:KW-0046',
            'format': 'tsv', 'fields': 'accession'}, timeout=30)
uniprot_amr_ids = set(l.split('\t')[0] for l in r_kw.text.strip().split('\n')[1:] if l)
overlap_kw46 = uniprot_amr_ids & fold_ids
n_amr_kw = df_fs_best['has_amr_kw'].sum()
n_with_pdb = (df_fs_best['pdb_refs'] != '').sum()

print(f'UniProt AMR keyword:     {n_amr_kw}/{len(df_fs_best)}')
print(f'KW-0046 overlap:         {len(overlap_kw46)}/{len(uniprot_amr_ids)}')
print(f'Hits with PDB cross-ref: {n_with_pdb}')
print('\n✓ Cell 10 complete')


## Cell 10b: Background GO Fetch (All 2,106 NEIG1 Proteins)

In [ ]:
# ── CELL 10b: Fetch GO + KEGG for ALL NEIG1 proteins (enrichment background) ──
# Required for statistically valid GO enrichment (Fisher's exact needs background)
# ~10 min with 12 workers | lighter fetch: GO + KEGG refs only

RESULTS = CONFIG['local_results_dir']
DATA    = CONFIG['local_data_dir']
BG_CACHE_PATH = f'{RESULTS}/uniprot_bg_cache.json'

if checkpoint_exists('uniprot_bg_cache.json'):
    load_checkpoint('uniprot_bg_cache.json', BG_CACHE_PATH)
    with open(BG_CACHE_PATH) as f:
        bg_cache = json.load(f)
    print(f'✓ Background GO cache loaded: {len(bg_cache):,} entries')
else:
    # All NEIG1 protein IDs from FASTA
    all_neig1_ids = list(load_fasta(
        f'{DATA}/neisseria/NEIG1.fasta').keys())
    print(f'Fetching GO+KEGG for {len(all_neig1_ids):,} NEIG1 proteins...')

    bg_cache = {}
    t0 = time.time()

    def fetch_go_only(uid, max_retries=3):
        """Lightweight fetch: GO terms + KEGG cross-refs only."""
        for attempt in range(max_retries):
            try:
                r = requests.get(
                    f'https://rest.uniprot.org/uniprotkb/{uid}.json',
                    timeout=20)
                if r.status_code == 429:
                    time.sleep((2 ** attempt) + random.uniform(0, 1))
                    continue
                if r.status_code != 200:
                    return uid, {}
                d = r.json()
                go_refs  = [x for x in d.get('dbReferences', [])
                            if x['type'] == 'GO']
                kegg_ids = [x['id'] for x in d.get('dbReferences', [])
                            if x['type'] == 'KEGG']
                time.sleep(0.1)
                return uid, {
                    'go':  [(x['id'],
                             x.get('properties', {}).get('term', ''),
                             x.get('properties', {}).get('source', ''))
                            for x in go_refs],
                    'kegg': kegg_ids,
                }
            except Exception as e:
                if attempt == max_retries - 1:
                    return uid, {'error': str(e)}
                time.sleep(2 ** attempt)
        return uid, {}

    with ThreadPoolExecutor(max_workers=12) as ex:
        futs = {ex.submit(fetch_go_only, uid): uid
                for uid in all_neig1_ids}
        for i, fut in enumerate(as_completed(futs)):
            uid, res = fut.result()
            bg_cache[uid] = res
            if (i+1) % 200 == 0:
                pct = (i+1)/len(all_neig1_ids)*100
                print(f'  {i+1:,}/{len(all_neig1_ids):,} ({pct:.0f}%)')

    elapsed = time.time() - t0
    n_with_go   = sum(1 for v in bg_cache.values() if v.get('go'))
    n_with_kegg = sum(1 for v in bg_cache.values() if v.get('kegg'))
    print(f'✓ Done in {elapsed:.0f}s')
    print(f'  Proteins with GO:   {n_with_go:,}')
    print(f'  Proteins with KEGG: {n_with_kegg:,}')

    with open(BG_CACHE_PATH, 'w') as f:
        json.dump(bg_cache, f)
    save_checkpoint(BG_CACHE_PATH, 'uniprot_bg_cache.json')

print('\n✓ Cell 10b complete')


## Cell 11: pLDDT + Expanded PDB Validation (25 structures + TM-score)

In [ ]:
# ── CELL 11: pLDDT + PDB validation (25 representative AMR structures) ──────────

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']

# ── pLDDT from CIF files ───────────────────────────────────────────────────
def get_plddt(uid):
    p = f'{DATA}/neisseria/AF-{uid}-F1-model_v6.cif.gz'
    try:
        with gzip.open(p, 'rt') as f:
            vals = [float(l.split()[-2]) for l in f
                    if l.startswith('ATOM') and ' CA ' in l]
            return sum(vals)/len(vals) if vals else None
    except:
        return None

print('Computing pLDDT scores (16 parallel workers)...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    plddt_map = dict(zip(df_fs_best['neig1_id'],
                         ex.map(get_plddt, df_fs_best['neig1_id'])))
df_fs_best['mean_plddt'] = df_fs_best['neig1_id'].map(plddt_map)
n70  = (df_fs_best['mean_plddt'] > 70).sum()
n90  = (df_fs_best['mean_plddt'] > 90).sum()
print(f'pLDDT >70: {n70}/{len(df_fs_best)} | >90: {n90}/{len(df_fs_best)} ({time.time()-t0:.0f}s)')

# ── 25 representative PDB structures — all major CARD gene families ────────
PDB_TEMPLATES = {
    # Beta-lactamases (4 classes)
    'TEM-1_ClassA':         '1BTL',
    'CTX-M-15_ClassA':      '4HBT',
    'NDM-1_ClassB':         '3Q6X',
    'AmpC_ClassC':          '1KE4',
    'OXA-10_ClassD':        '1K54',
    # Efflux pumps
    'AcrB_RND':             '4DX5',
    'MexB_RND':             '2V50',
    'MacB_ABC':             '5GKO',
    'EmrD_MFS':             '2GFP',
    # Aminoglycoside modifying enzymes
    'AAC6_AME':             '1B87',
    'APH3_AME':             '1L8T',
    # Ribosomal protection
    'TetM_RibProt':         '4V9P',
    # rRNA methyltransferases
    'ErmC_Methyltrans':     '1QAM',
    # Vancomycin resistance
    'VanA_Ligase':          '1E4E',
    'VanH_Dehydrogenase':   '4MFD',
    # Other mechanisms
    'FosA_Fosfomycin':      '1LQP',
    'CAT_Chloramphenicol':  '3CLA',
    'Sul1_Sulfonamide':     '1AJ0',
    'DHFR_Ia':              '3DAR',
    'MurA_TargetAlt':       '1A2N',
    'QnrB_Fluoroquinolone': '2XTW',
    'MCR1_PMR':             '5GOB',
    'MphA_Macrolide':       '5IGR',
    'BlaI_Regulator':       '1SD3',
    'TtgR_EffluxReg':       '2UXI',
}

def dl_pdb(args):
    name, pdb_id = args
    out = f'{DATA}/card_pdb/{name}_{pdb_id}.cif'
    if Path(out).exists(): return True
    try:
        r = requests.get(f'https://files.rcsb.org/download/{pdb_id}.cif', timeout=30)
        if r.status_code == 200:
            Path(out).write_text(r.text)
            return True
    except: pass
    time.sleep(0.3)
    return False

Path(f'{DATA}/card_pdb').mkdir(exist_ok=True)
print(f'Downloading {len(PDB_TEMPLATES)} PDB structures...')
with ThreadPoolExecutor(max_workers=10) as ex:
    results = list(ex.map(dl_pdb, PDB_TEMPLATES.items()))
n_pdb_dl = sum(results)
print(f'Downloaded: {n_pdb_dl}/{len(PDB_TEMPLATES)} PDB structures')

# ── Foldseek real structure-structure search with TM-score ────────────────
print('Running Foldseek real structure search + TM-score...')
t0 = time.time()
pdb_out = f'{RESULTS}/pdb_vs_neig1.tsv'
subprocess.run(
    f'foldseek easy-search {DATA}/card_pdb/ {DATA}/dbs/neig1_db '
    f'{pdb_out} {CONFIG["local_tmp_dir"]}/foldseek '
    f'--format-output query,target,pident,evalue,qlen,tlen,alnlen,bits,tmscore '
    f'-e 0.001 --threads 4 -v 1',
    shell=True, env=os.environ)
print(f'✓ PDB search complete ({time.time()-t0:.0f}s)')

# Parse PDB results
df_pdb_raw = pd.read_csv(pdb_out, sep='\t',
    names=['query','target','pident','evalue','qlen','tlen','alnlen','bits','tmscore'])
df_pdb_raw['neig1_id'] = df_pdb_raw['target'].apply(
    lambda x: x.split('-')[1] if x.startswith('AF-') else x)

# Strict filter
df_pdb = df_pdb_raw[df_pdb_raw['evalue'] < 1e-5].copy()
pdb_hits = set(df_pdb['neig1_id'])

# TM-score statistics
print(f'\nTM-score distribution (threshold e<1e-5):')
print(f'  Mean TM-score:    {df_pdb["tmscore"].mean():.3f}')
print(f'  Median TM-score:  {df_pdb["tmscore"].median():.3f}')
print(f'  TM > 0.5 (same fold):      {(df_pdb["tmscore"] > 0.5).sum()} hits')
print(f'  TM > 0.7 (similar struct): {(df_pdb["tmscore"] > 0.7).sum()} hits')

# Concordance metrics
# A: Foldseek hits also found by real structure search
conc_A = len(pdb_hits & fold_ids) / max(len(pdb_hits), 1) * 100
# B: Foldseek hits with UniProt PDB cross-reference
hits_with_pdb_xref = set(df_fs_best[df_fs_best['pdb_refs'] != '']['neig1_id'])
conc_B = len(hits_with_pdb_xref) / len(fold_ids) * 100

print(f'\nPDB validation:')
print(f'  Templates used:                 {n_pdb_dl} structures ({len(PDB_TEMPLATES)} families)')
print(f'  Real structure search hits:     {len(pdb_hits)}')
print(f'  Concordance A (real struct):    {len(pdb_hits & fold_ids)} ({conc_A:.1f}%)')
print(f'  Concordance B (UniProt PDB xref): {len(hits_with_pdb_xref)} ({conc_B:.1f}%)')

CONFIG['pdb_validation'] = {
    'n_templates': n_pdb_dl,
    'n_families': len(PDB_TEMPLATES),
    'concordance_A_pct': round(conc_A, 1),
    'concordance_B_pct': round(conc_B, 1),
    'tmscore_mean': round(float(df_pdb['tmscore'].mean()), 3),
    'tmscore_gt05': int((df_pdb['tmscore'] > 0.5).sum()),
}
print('\n✓ Cell 11 complete')


## Cell 12: Coverage-Identity Structural Hit Categories

In [ ]:
# ── CELL 12: Coverage-identity matrix — novel analysis ────────────────────────
import pandas as pd
import numpy as np

# Build 2D distribution: sequence identity vs target coverage
# This characterises the structural hit landscape

# Define bins
pident_bins  = [0, 30, 50, 70, 90, 100]
pident_labels = ['<30%\ncryptic', '30-50%\nremote', '50-70%\nmoderate',
                  '70-90%\nclose', '>90%\nnear-id.']
tcov_bins    = [0, 0.3, 0.5, 0.7, 0.9, 1.01]
tcov_labels  = ['<30%', '30-50%', '50-70%', '70-90%', '90-100%']

df_fs_best['pident_cat'] = pd.cut(df_fs_best['pident'],
    bins=pident_bins, labels=pident_labels, right=True)
df_fs_best['tcov_cat']   = pd.cut(df_fs_best['tcov'],
    bins=tcov_bins, labels=tcov_labels, right=True)

# Count matrix
matrix = df_fs_best.groupby(['pident_cat', 'tcov_cat'], observed=True).size().unstack(fill_value=0)

print('Coverage-Identity matrix (rows=identity, cols=target coverage):')
print(matrix.to_string())

# Summary statistics
print(f'\nStructural hit tier summary:')
print(f'  Tier 1 (tcov <50%, cryptic): {df_fs_best[(df_fs_best["tcov"] < 0.5)].shape[0]} proteins (partial domain match)')
print(f'  Tier 2 (tcov 50-70%):        {df_fs_best[(df_fs_best["tcov"] >= 0.5) & (df_fs_best["tcov"] < 0.7)].shape[0]} proteins (domain-level)')
print(f'  Tier 3 (tcov 70-90%):        {df_fs_best[(df_fs_best["tcov"] >= 0.7) & (df_fs_best["tcov"] < 0.9)].shape[0]} proteins (near full-length)')
print(f'  Tier 4 (tcov >90%):          {df_fs_best[(df_fs_best["tcov"] >= 0.9)].shape[0]} proteins (full-length match)')

CONFIG['coverage_identity_matrix'] = matrix.to_dict()
print('\n✓ Cell 12 complete')


## Cell 13: KEGG + GO Pathway Analysis

In [ ]:
# ── CELL 13: GO enrichment + KEGG pathway completeness ────────────────────────
# GO: Fisher's exact test + BH FDR (GOATOOLS framework)
# KEGG: UniProt cross-ref mapping + pathway completeness + enrichment

!pip install -q goatools
from collections import Counter, defaultdict
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np
import random, time

RESULTS = CONFIG['local_results_dir']

# ── 1. Build GO annotation sets ───────────────────────────────────────────
print('Building GO annotation sets...')

study_ids      = set(df_fs_best['neig1_id'])
population_ids = set(bg_cache.keys())

# GO term → proteins mapping for background and study
go2proteins_bg    = defaultdict(set)
go2proteins_study = defaultdict(set)
go2term           = {}

for uid, info in bg_cache.items():
    for go_id, go_term, go_src in info.get('go', []):
        go2proteins_bg[go_id].add(uid)
        go2term[go_id] = go_term
        if uid in study_ids:
            go2proteins_study[go_id].add(uid)

n_study = len(study_ids)
n_bg    = len(population_ids)
print(f'Study: {n_study} | Background: {n_bg}')
print(f'GO terms in background: {len(go2proteins_bg)}')

# ── 2. Fisher's exact test per GO term ────────────────────────────────────
print('\nRunning Fisher\'s exact test (min 3 study proteins)...')

go_results = []
for go_id, study_proteins in go2proteins_study.items():
    k = len(study_proteins)
    if k < 3:
        continue
    M = len(go2proteins_bg[go_id])
    # 2x2 table: hits_with / hits_without / bg_with / bg_without
    table = [[k, n_study - k],
             [M - k, n_bg - n_study - (M - k)]]
    try:
        _, pval = stats.fisher_exact(table, alternative='greater')
    except:
        pval = 1.0

    go_term  = go2term.get(go_id, '')
    go_results.append({
        'go_id':           go_id,
        'go_term':         go_term,
        'namespace':       ('BP' if go_term.startswith('P:') else
                             'MF' if go_term.startswith('F:') else
                             'CC' if go_term.startswith('C:') else 'other'),
        'go_term_clean':   go_term[2:] if len(go_term)>2 and go_term[1]==':' else go_term,
        'n_study':         k,
        'n_background':    M,
        'study_ratio':     f'{k}/{n_study}',
        'bg_ratio':        f'{M}/{n_bg}',
        'fold_enrichment': (k/n_study) / max(M/n_bg, 1e-10),
        'pvalue':          pval,
        'hit_proteins':    ';'.join(sorted(study_proteins)),
    })

df_go = pd.DataFrame(go_results)

if len(df_go) > 0:
    _, fdr, _, _ = multipletests(df_go['pvalue'], method='fdr_bh')
    df_go['fdr'] = fdr
    df_go = df_go.sort_values('fdr')
    df_go_sig = df_go[df_go['fdr'] < 0.05].copy()

    print(f'Total GO terms tested: {len(df_go)}')
    print(f'Significant (FDR<0.05): {len(df_go_sig)}')
    print(f'\nTop enriched GO terms:')
    for _, row in df_go_sig.head(15).iterrows():
        print(f'  [{row["namespace"]}] {row["go_id"]} | '              f'{row["go_term_clean"][:45]:<45} | '              f'n={row["n_study"]:3d} | FE={row["fold_enrichment"]:.1f}x | '              f'FDR={row["fdr"]:.2e}')
else:
    df_go_sig = pd.DataFrame()
    print('No GO terms met minimum threshold')

df_go.to_csv(f'{RESULTS}/go_enrichment.tsv', sep='\t', index=False)

# ── 3. KEGG mapping via UniProt cross-refs (ngo:NGO_XXXX IDs) ────────────
print('\nBuilding KEGG map via UniProt cross-references...')

neig1_to_kegg  = {}
for uid, info in bg_cache.items():
    ngo_ids = [k for k in info.get('kegg', []) if k.startswith('ngo:')]
    if ngo_ids:
        neig1_to_kegg[uid] = ngo_ids

hit_kegg_map = {uid: ids for uid, ids in neig1_to_kegg.items()
                if uid in study_ids}
bg_kegg_flat  = set(kid for ids in neig1_to_kegg.values() for kid in ids)
hit_kegg_flat = set(kid for ids in hit_kegg_map.values() for kid in ids)

print(f'NEIG1 proteins with NGO KEGG ID: {len(neig1_to_kegg)}')
print(f'Foldseek hits with NGO KEGG ID:  {len(hit_kegg_map)}')

def get_pathways_for_gene(kegg_gene_id):
    try:
        r = requests.get(f'https://rest.kegg.jp/get/{kegg_gene_id}', timeout=8)
        pws, in_pw = [], False
        for line in r.text.split('\n'):
            if line.startswith('PATHWAY'):
                in_pw = True
                parts = line.split()
                if len(parts) >= 3:
                    pws.append((parts[1], ' '.join(parts[2:])))
            elif in_pw and line.startswith('            '):
                parts = line.strip().split()
                if len(parts) >= 2:
                    pws.append((parts[0], ' '.join(parts[1:])))
            elif in_pw and not line.startswith(' '):
                in_pw = False
        return pws
    except:
        return []

def fetch_kegg_for_hit(uid):
    all_pws = []
    for kid in hit_kegg_map.get(uid, []):
        all_pws.extend(get_pathways_for_gene(kid))
        time.sleep(0.2 + random.uniform(0, 0.05))
    return uid, all_pws

print(f'Fetching KEGG pathways for {len(hit_kegg_map)} proteins...')
t0 = time.time()
kegg_map = {}
with ThreadPoolExecutor(max_workers=3) as ex:
    for uid, pws in ex.map(fetch_kegg_for_hit, list(hit_kegg_map.keys())):
        if pws:
            kegg_map[uid] = pws
print(f'✓ KEGG mapping done ({time.time()-t0:.0f}s) | {len(kegg_map)} proteins mapped')

# ── 4. Pathway completeness + enrichment ──────────────────────────────────
print('\nComputing pathway completeness...')

def get_all_ngo_pathways():
    try:
        r = requests.get('https://rest.kegg.jp/list/pathway/ngo', timeout=15)
        return {line.split('\t')[0].replace('path:',''): line.split('\t')[1].strip()
                for line in r.text.strip().split('\n') if '\t' in line}
    except:
        return {}

def get_pathway_genes(pw_id):
    try:
        r = requests.get(f'https://rest.kegg.jp/get/{pw_id}', timeout=10)
        genes, in_gene = [], False
        for line in r.text.split('\n'):
            if line.startswith('GENE'):
                in_gene = True
                parts = line.split()
                if len(parts) >= 2: genes.append(f'ngo:{parts[1]}')
            elif in_gene and line.startswith('            '):
                parts = line.strip().split()
                if parts: genes.append(f'ngo:{parts[0]}')
            elif in_gene and not line.startswith(' '): in_gene = False
        return genes
    except:
        return []

ngo_pathways = get_all_ngo_pathways()
print(f'Total NGO pathways in KEGG: {len(ngo_pathways)}')

# Pathways containing our hits
pathways_with_hits = set(
    pw_id for pws in kegg_map.values() for pw_id, _ in pws)
print(f'Pathways with Foldseek hits: {len(pathways_with_hits)}')

completeness_rows = []
for pw_id in list(pathways_with_hits)[:40]:
    pw_genes = get_pathway_genes(pw_id)
    if not pw_genes: continue
    n_total  = len(pw_genes)
    n_in_hit = len(set(pw_genes) & hit_kegg_flat)
    n_in_bg  = len(set(pw_genes) & bg_kegg_flat)
    # Fisher's exact enrichment
    table = [[n_in_hit, len(hit_kegg_flat) - n_in_hit],
             [n_in_bg - n_in_hit,
              len(bg_kegg_flat) - len(hit_kegg_flat) - (n_in_bg - n_in_hit)]]
    try:
        _, pval = stats.fisher_exact(table, alternative='greater')
    except:
        pval = 1.0
    completeness_rows.append({
        'pathway_id':          pw_id,
        'pathway_name':        ngo_pathways.get(pw_id, pw_id),
        'n_pathway_genes':     n_total,
        'n_in_hits':           n_in_hit,
        'n_in_proteome':       n_in_bg,
        'completeness_hits':   round(n_in_hit/max(n_total,1), 3),
        'completeness_bg':     round(n_in_bg/max(n_total,1), 3),
        'pvalue':              pval,
    })
    time.sleep(0.2)

df_kegg = pd.DataFrame(completeness_rows)
if len(df_kegg) > 0:
    _, fdr_kegg, _, _ = multipletests(df_kegg['pvalue'], method='fdr_bh')
    df_kegg['fdr'] = fdr_kegg
    df_kegg = df_kegg.sort_values('completeness_hits', ascending=False)
    print('\nTop pathways by completeness in Foldseek hits:')
    for _, row in df_kegg.head(10).iterrows():
        print(f'  {row["pathway_id"]} | {row["pathway_name"][:40]:<40} | '              f'completeness={row["completeness_hits"]:.0%} '              f'({row["n_in_hits"]}/{row["n_pathway_genes"]}) | '              f'FDR={row["fdr"]:.2e}')
    df_kegg.to_csv(f'{RESULTS}/kegg_completeness.tsv', sep='\t', index=False)

# Update df_fs_best kegg_pathways column
kegg_map_names = {uid: [n for _, n in pws] for uid, pws in kegg_map.items()}
df_fs_best['kegg_pathways'] = df_fs_best['neig1_id'].map(
    lambda x: ';'.join(kegg_map_names.get(x, [])))

with open(f'{RESULTS}/kegg_pathways.json', 'w') as f:
    json.dump({uid: [[pid, pn] for pid, pn in pws]
               for uid, pws in kegg_map.items()}, f, indent=2)

pw_counter = Counter([n for pws in kegg_map.values() for _, n in pws])

print(f'\n✓ Cell 13 complete')
print(f'  GO significant (FDR<0.05): {len(df_go_sig)}')
print(f'  KEGG proteins mapped:      {len(kegg_map)}')
print(f'  Pathways analysed:         {len(completeness_rows)}')


## Cell 14: UMAP — AMR Embedding Space Analysis

In [ ]:
# ── CELL 14: UMAP of AMR protein embedding space ──────────────────────────────
# Novel analysis: are ESM2 clusters biologically meaningful for AMR?
# UMAP colored 3 ways: gene family, mechanism, drug class
# NEIG1 hits overlaid as a separate layer

!pip install -q umap-learn
import umap
import numpy as np
import json

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']

# Load AMR query embeddings (computed during ESM2 cell)
# Re-embed AMR queries for UMAP (use same model, same seed)
print('Embedding AMR query sequences for UMAP...')
t0 = time.time()
amr_seqs_for_umap = load_fasta(QUERY_FASTA, id_parse='first')
amr_ids_umap  = list(amr_seqs_for_umap.keys())
amr_embs_umap = embed_batch(list(amr_seqs_for_umap.values()))
print(f'✓ AMR embeddings: {amr_embs_umap.shape} ({time.time()-t0:.0f}s)')

# Also get NEIG1 hit embeddings (subset of neig1_embs)
hit_indices = [neig1_ids.index(uid) for uid in fold_ids if uid in neig1_ids]
neig1_hit_embs = neig1_embs[hit_indices]
neig1_hit_ids  = [neig1_ids[i] for i in hit_indices]

# Combine for UMAP: AMR queries + NEIG1 hits
combined_embs = np.vstack([amr_embs_umap.numpy(), neig1_hit_embs.numpy()])
n_amr   = len(amr_ids_umap)
n_neig1 = len(neig1_hit_ids)
labels  = ['AMR_query'] * n_amr + ['NEIG1_hit'] * n_neig1

print(f'UMAP input: {n_amr:,} AMR queries + {n_neig1} NEIG1 hits = {len(combined_embs):,} total')

# Fit UMAP — reproducible parameters
print('Fitting UMAP (cosine metric, seed=142857)...')
t0 = time.time()
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric='cosine',        # critical: matches our similarity computation
    random_state=CONFIG['random_seed'],
    verbose=False
)
embedding = reducer.fit_transform(combined_embs)
print(f'✓ UMAP fit in {time.time()-t0:.0f}s')

# Build annotation dataframes
# For AMR queries: get CARD metadata
df_amr_umap = pd.DataFrame({
    'uid':       amr_ids_umap,
    'umap_x':    embedding[:n_amr, 0],
    'umap_y':    embedding[:n_amr, 1],
    'source':    'AMR_query',
    'gene_family': [aro_meta.get(aro_lookup.get(
        uid.replace('[CARD]','').split('|')[1] if '|' in uid else uid.replace('[CARD]',''), ''),
        {}).get('gene_family', 'Unknown')[:40] for uid in amr_ids_umap],
    'mechanism': [aro_meta.get(aro_lookup.get(
        uid.replace('[CARD]','').split('|')[1] if '|' in uid else uid.replace('[CARD]',''), ''),
        {}).get('mechanism', 'Unknown') for uid in amr_ids_umap],
    'drug_class': [aro_meta.get(aro_lookup.get(
        uid.replace('[CARD]','').split('|')[1] if '|' in uid else uid.replace('[CARD]',''), ''),
        {}).get('drug_class', 'Unknown')[:30] for uid in amr_ids_umap],
})

# For NEIG1 hits: map to their annotations
df_neig1_umap = pd.DataFrame({
    'uid':        neig1_hit_ids,
    'umap_x':     embedding[n_amr:, 0],
    'umap_y':     embedding[n_amr:, 1],
    'source':     'NEIG1_hit',
    'gene_family': df_fs_best.set_index('neig1_id')['gene_family'].reindex(neig1_hit_ids).fillna('Unknown').values,
    'mechanism':   df_fs_best.set_index('neig1_id')['mechanism'].reindex(neig1_hit_ids).fillna('Unknown').values,
    'drug_class':  df_fs_best.set_index('neig1_id')['drug_class'].reindex(neig1_hit_ids).fillna('Unknown').values,
})

df_umap = pd.concat([df_amr_umap, df_neig1_umap], ignore_index=True)
df_umap.to_csv(f'{RESULTS}/umap_embedding.tsv', sep='\t', index=False)

# Quick cluster quality check
from sklearn.metrics import silhouette_score
# Check if ESM2 clusters align with mechanism
mechanism_labels = df_amr_umap['mechanism'].values
unique_mechs = [m for m in set(mechanism_labels) if mechanism_labels.tolist().count(m) > 5]
if len(unique_mechs) > 1:
    mask = [m in unique_mechs for m in mechanism_labels]
    sub_embs   = embedding[:n_amr][mask]
    sub_labels = [mechanism_labels[i] for i in range(len(mask)) if mask[i]]
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    encoded = le.fit_transform(sub_labels)
    if len(set(encoded)) > 1 and len(sub_embs) > 10:
        sil = silhouette_score(sub_embs, encoded)
        print(f'\nESM2 cluster quality (silhouette by mechanism): {sil:.3f}')
        print(f'  >0.25 = reasonable, >0.5 = good clustering')
        CONFIG['esm2_silhouette_by_mechanism'] = round(float(sil), 3)

print(f'\nUMAP saved: {len(df_umap):,} points')
print(f'  AMR queries: {n_amr:,}')
print(f'  NEIG1 hits:  {n_neig1}')
print('\n✓ Cell 14 complete')


## Cell 15: Five-way Comparison Table

In [ ]:
# ── CELL 15: Five-way comparison table ────────────────────────────────────────
import pandas as pd

DATA    = CONFIG['local_data_dir']
RESULTS = CONFIG['local_results_dir']
N_DB    = int(open(f'{DATA}/dbs/neig1_db.lookup').read().count('\n'))

excl_seq = fold_ids - mmseqs_ids - diamond_ids
excl_all = fold_ids - mmseqs_ids - diamond_ids - esm2_ids - amrfinder_ids_full

# Exclusive vs all 5 methods
excl_vs_5 = fold_ids - mmseqs_ids - diamond_ids - esm2_ids - amrfinder_ids_all

print('=' * 68)
print('AMRFOLD v3.0 — FINAL VERIFIED RESULTS')
print('=' * 68)
print(f'Organism:  {CONFIG["organism_name"]}')
print(f'Query DB:  CARD v{CONFIG["card_version"]} + UniProt SwissProt KW-0046 bacteria')
print(f'           → {N_COMBINED:,} non-redundant protein sequences')
print(f'Target:    AFDB v6 | {N_DB:,} structures')
print(f'Seed:      {CONFIG["random_seed"]} (cyclic number 1/7)')
print(f'Date:      {datetime.now().strftime("%Y-%m-%d")}')
print()
print(f'{"Method":<45} {"Hits":>6} {"% proteome":>11}')
print('-' * 68)
for label, ids in [
    ('MMseqs2 (sequence alignment)',         mmseqs_ids),
    ('DIAMOND BLASTp (sensitive)',           diamond_ids),
    ('ESM2 pLM (p=1e-5, stat. calibrated)', esm2_ids),
    ('AMRFinderPlus HMM (BLAST+HMM)',        amrfinder_ids_full),
    ('AMRFinderPlus HMM (all+PARTIAL)',      amrfinder_ids_all),
    ('Foldseek+ProstT5 [AMRfold]',          fold_ids),
]:
    print(f'{label:<45} {len(ids):>6,} {len(ids)/N_DB*100:>10.1f}%')
print(f'{"PDB structural validation":<45} {len(pdb_hits):>6,} {"("+str(n_pdb_dl)+" templates)":>11}')
print('=' * 68)
print()
print(f'Foldseek vs MMseqs2:          {len(fold_ids)/max(len(mmseqs_ids),1):.1f}x')
print(f'Foldseek vs DIAMOND:          {len(fold_ids)/max(len(diamond_ids),1):.1f}x')
print(f'Foldseek vs ESM2:             {len(fold_ids)/max(len(esm2_ids),1):.1f}x')
print(f'Foldseek vs AMRFinder (full): {len(fold_ids)/max(len(amrfinder_ids_full),1):.1f}x')
print()
print(f'Exclusive vs seq methods:     {len(excl_seq):,}')
print(f'Exclusive vs all 4 methods:   {len(excl_all):,} ({len(excl_all)/len(fold_ids)*100:.1f}%)')
print(f'Exclusive vs all 5 methods:   {len(excl_vs_5):,} ({len(excl_vs_5)/len(fold_ids)*100:.1f}%)')
print()
print(f'Quality metrics:')
print(f'  Cryptic (<30% identity):    {df_fs_best["cryptic"].sum():,} ({df_fs_best["cryptic"].mean()*100:.1f}%)')
print(f'  High pLDDT (>70):           {n70}/{len(df_fs_best)} ({n70/len(df_fs_best)*100:.1f}%)')
print(f'  No human homolog:           {len(no_human):,} ({len(no_human)/len(fold_ids)*100:.1f}%)')
print(f'  UniProt AMR keyword:        {n_amr_kw}/{len(df_fs_best)} ({n_amr_kw/len(df_fs_best)*100:.1f}%)')
print(f'  PDB concordance (real str): {conc_A:.1f}%')
print(f'  PDB UniProt cross-ref:      {conc_B:.1f}%')
print(f'  Multi-domain proteins:      {n_double+n_triple:,}')
print()
print(f'AMRFinderPlus concordance:')
print(f'  AMRFinder full ∩ Foldseek: {len(amrfinder_ids_full & fold_ids)} ({len(amrfinder_ids_full & fold_ids)/max(len(amrfinder_ids_full),1)*100:.1f}% of AMRFinder)')
print(f'  Foldseek only (not AMRFinder): {len(fold_ids - amrfinder_ids_all)} — novel structural hits')

# Final integrity checks
print('\nIntegrity checks:')
assert len(fold_ids) > len(mmseqs_ids),  'FAIL: Foldseek <= MMseqs2'
assert len(fold_ids) > len(diamond_ids), 'FAIL: Foldseek <= DIAMOND'
assert len(diamond_ids) > 0,             'FAIL: DIAMOND = 0'
assert df_fs_best['cryptic'].mean() > 0.5, 'WARN: <50% cryptic'
print('  ✓ All checks passed')
print('\n✓ Cell 15 complete')


## Cell 16: Save All Results + Provenance

In [ ]:
# ── CELL 16: Save all results with complete provenance ────────────────────────
import json
from datetime import datetime

RESULTS = CONFIG['local_results_dir']

# Save annotated best hits
df_fs_best.to_csv(f'{RESULTS}/neig1_amr_best.tsv', sep='\t', index=False)
# Save all hits (for multi-domain, full analysis)
df_fs_all.to_csv(f'{RESULTS}/neig1_amr_all.tsv', sep='\t', index=False)
# Save cluster membership
shutil.copy(f'{CONFIG["local_data_dir"]}/combined_amr_clust_cluster.tsv',
            f'{RESULTS}/cluster_membership.tsv')

# Complete provenance
provenance = {
    'tool':              'AMRfold v3.0',
    'run_date':          CONFIG['run_start'],
    'run_complete':      datetime.now().isoformat(),
    'random_seed':       CONFIG['random_seed'],

    # Databases
    'card_version':      CONFIG['card_version'],
    'card_accessed':     CONFIG['card_accessed'],
    'swissprot_amr_bacteria_count': CONFIG['swissprot_amr_bacteria_count'],
    'swissprot_accessed': CONFIG['swissprot_accessed'],
    'afdb_version':      'v6',
    'afdb_accessed':     CONFIG['afdb_accessed'],
    'amrfinderplus_db_date': CONFIG['amrfinder_db_date'],

    # Tools
    'tool_versions':     CONFIG['tool_versions'],
    'esm2_model':        CONFIG['esm2_model'],

    # Query
    'n_card_sequences':  N_CARD,
    'n_swissprot_sequences': CONFIG['swissprot_amr_bacteria_count'],
    'n_combined_nr':     N_COMBINED,
    'cluster_identity':  CONFIG['cluster_identity'],
    'cluster_stats':     CONFIG.get('cluster_stats', {}),

    # Target
    'organism':          CONFIG['organism_name'],
    'uniprot_proteome':  CONFIG['uniprot_proteome'],
    'n_structures':      N_DB,

    # Filters
    'filters': {
        'evalue':   CONFIG['evalue_threshold'],
        'qcov_min': CONFIG['qcov_min'],
        'tcov_min': CONFIG['tcov_min'],
        'pident_cryptic': CONFIG['pident_cryptic'],
    },

    # ESM2
    'esm2_null_distribution': CONFIG.get('esm2_null_dist', ''),
    'esm2_null_mean':         CONFIG.get('esm2_null_mean', 0),
    'esm2_null_std':          CONFIG.get('esm2_null_std', 0),
    'esm2_threshold':         CONFIG.get('esm2_threshold', 0),
    'esm2_pvalue':            CONFIG['esm2_pvalue'],
    'esm2_silhouette':        CONFIG.get('esm2_silhouette_by_mechanism', None),

    # PDB validation
    'pdb_validation':         CONFIG.get('pdb_validation', {}),

    # Results
    'results': {
        'foldseek':              len(fold_ids),
        'mmseqs2':               len(mmseqs_ids),
        'diamond':               len(diamond_ids),
        'esm2':                  len(esm2_ids),
        'amrfinderplus_full':    len(amrfinder_ids_full),
        'amrfinderplus_all':     len(amrfinder_ids_all),
        'cryptic':               int(df_fs_best['cryptic'].sum()),
        'high_plddt_70':         int(n70),
        'high_plddt_90':         int(n90),
        'no_human_homolog':      len(no_human),
        'exclusive_vs_seq':      len(excl_seq),
        'exclusive_vs_4method':  len(excl_all),
        'exclusive_vs_5method':  len(excl_vs_5),
        'multidomain_proteins':  n_double + n_triple,
        'uniprot_amr_keyword':   int(n_amr_kw),
        'pdb_concordance_A_pct': round(conc_A, 1),
        'pdb_concordance_B_pct': round(conc_B, 1),
    },

    # Runtimes
    'runtimes_seconds': {
        'foldseek':       CONFIG.get('foldseek_runtime_s', None),
        'amrfinderplus':  CONFIG.get('amrfinder_runtime_s', None),
    },

    # Limitations
    'limitations': [
        'ProstT5 asymmetric search: predicted 3Di (query) vs real AF2 3Di (target) — validated vs 25 PDB structures',
        'Acquired resistance genes only — chromosomal point mutations (penA, gyrA, parC) require CARD variant model',
        'ESM2 threshold statistically calibrated from NEIG1-NEIG1 null distribution (p=1e-5), not e-value',
        'KEGG mapping partial — only UniProt gene-name-resolved proteins with NGO KEGG entries',
        'AMRFinderPlus run without --organism flag — consistent with acquired-gene-only scope',
        'Coverage filter tcov>=0.3 is minimum detectability threshold — see coverage-identity tier analysis',
    ]
}

with open(f'{RESULTS}/provenance.json', 'w') as f:
    json.dump(provenance, f, indent=2)

# Save files listing
print('Saved results:')
for fname in sorted(Path(RESULTS).glob('*.tsv')) + sorted(Path(RESULTS).glob('*.json')):
    size = fname.stat().st_size / 1024
    print(f'  {fname.name} ({size:.0f} KB)')
print('\n✓ Cell 16 complete')


## Cell 17: Individual Publication-Quality Figures

In [ ]:
# ── CELL 17: Individual figures — 300 DPI, colorblind-safe, panel letters ────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

!pip install -q matplotlib-venn seaborn
from matplotlib_venn import venn2
import seaborn as sns

FIGS = CONFIG['local_figures_dir']
Path(FIGS).mkdir(exist_ok=True)

# ── Colorblind-safe palette (Wong 2011) ───────────────────────────────────
CB = {
    'black':   '#000000',
    'orange':  '#E69F00',
    'skyblue': '#56B4E9',
    'green':   '#009E73',
    'yellow':  '#F0E442',
    'blue':    '#0072B2',
    'red':     '#D55E00',
    'pink':    '#CC79A7',
}
CB_LIST = list(CB.values())

# Global style
plt.rcParams.update({
    'font.family':     'DejaVu Sans',
    'font.size':       10,
    'axes.titlesize':  11,
    'axes.labelsize':  10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':      300,
    'savefig.dpi':     300,
    'savefig.bbox':    'tight',
    'savefig.format':  'png',
})

def add_panel_label(ax, label, fontsize=13):
    ax.text(-0.12, 1.05, label, transform=ax.transAxes,
            fontsize=fontsize, fontweight='bold', va='top', ha='right')

def save_fig(fig, name):
    path = f'{FIGS}/{name}.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved: {name}.png')

# ── Figure A: Five-method comparison bar ─────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
methods = ['MMseqs2\n(seq.)', 'DIAMOND\n(seq.)', 'ESM2\n(pLM)', 
           'AMRFinder\n(HMM)', 'Foldseek\n+ProstT5']
counts  = [len(mmseqs_ids), len(diamond_ids), len(esm2_ids),
           len(amrfinder_ids_full), len(fold_ids)]
colors  = [CB['blue'], CB['blue'], CB['skyblue'], CB['orange'], CB['red']]
bars = ax.bar(methods, counts, color=colors, edgecolor='white', width=0.65, linewidth=0.8)
for b, n in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 3,
            str(n), ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_ylabel('NEIG1 proteins detected', fontsize=10)
ax.set_title(f'AMR detection by search method\n({CONFIG["organism_name"].split()[0]} {CONFIG["organism_name"].split()[1]} FA 1090, strict filters: e<{CONFIG["evalue_threshold"]:.0e}, qcov≥{CONFIG["qcov_min"]}, tcov≥{CONFIG["tcov_min"]})',
             fontsize=10)
ax.set_ylim(0, max(counts) * 1.18)
add_panel_label(ax, 'A')
legend_elements = [
    mpatches.Patch(color=CB['blue'],    label='Sequence alignment'),
    mpatches.Patch(color=CB['skyblue'], label='Sequence embedding (pLM)'),
    mpatches.Patch(color=CB['orange'],  label='Profile HMM'),
    mpatches.Patch(color=CB['red'],     label='Structure (this work)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper left')
plt.tight_layout()
save_fig(fig, 'A_method_comparison')

# ── Figure B: Venn — Foldseek vs DIAMOND ─────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
venn2(subsets=(len(diamond_ids - fold_ids), len(fold_ids - diamond_ids),
               len(diamond_ids & fold_ids)),
      set_labels=('DIAMOND\n(sequence)', 'Foldseek\n(structure)'),
      set_colors=(CB['blue'], CB['red']), alpha=0.6, ax=ax)
ax.set_title('Overlap: Foldseek vs best\nsequence method (DIAMOND)', fontsize=11)
add_panel_label(ax, 'B')
plt.tight_layout()
save_fig(fig, 'B_venn_foldseek_diamond')

# ── Figure C: Sequence identity histogram ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_fs_best['pident'], bins=30, color=CB['red'], edgecolor='white',
        alpha=0.85, linewidth=0.5)
ax.axvline(30, color='black', linestyle='--', lw=1.5, label='30% threshold (cryptic)')
ax.set_xlabel('Sequence identity to query (%)', fontsize=10)
ax.set_ylabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title('Sequence identity distribution of structural AMR hits', fontsize=11)
ax.legend(fontsize=9)
n_crypt = df_fs_best['cryptic'].sum()
ax.text(0.98, 0.97,
        f'Cryptic (<30%): {n_crypt} ({n_crypt/len(df_fs_best)*100:.0f}%)',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=9, color=CB['red'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=CB['red'], alpha=0.8))
add_panel_label(ax, 'C')
plt.tight_layout()
save_fig(fig, 'C_sequence_identity')

# ── Figure D: pLDDT distribution ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_fs_best['mean_plddt'].dropna(), bins=25,
        color=CB['green'], edgecolor='white', alpha=0.85, linewidth=0.5)
ax.axvline(70, color=CB['orange'], linestyle='--', lw=1.5, label='pLDDT > 70 (high confidence)')
ax.axvline(90, color=CB['red'],    linestyle='--', lw=1.5, label='pLDDT > 90 (very high)')
ax.set_xlabel('Mean pLDDT score', fontsize=10)
ax.set_ylabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title('AlphaFold2 model confidence for structural AMR hits', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.02, 0.97,
        f'>70: {n70} ({n70/len(df_fs_best)*100:.0f}%)\n>90: {n90} ({n90/len(df_fs_best)*100:.0f}%)',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=CB['green'], alpha=0.8))
add_panel_label(ax, 'D')
plt.tight_layout()
save_fig(fig, 'D_plddt_distribution')

# ── Figure E: Top AMR gene families ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
top = df_fs_best['gene_family'].value_counts().head(12)
# Truncate long names intelligently
def truncate_label(s, maxlen=50):
    return s[:maxlen] + '...' if len(s) > maxlen else s
labels = [truncate_label(l) for l in top.index[::-1]]
bars = ax.barh(labels, top.values[::-1],
               color=CB['pink'], edgecolor='white', linewidth=0.5)
for b, n in zip(bars, top.values[::-1]):
    ax.text(b.get_width() + 0.2, b.get_y() + b.get_height()/2,
            str(n), va='center', ha='left', fontsize=8)
ax.set_xlabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title('Top AMR gene families (best structural hit per protein)', fontsize=11)
ax.tick_params(axis='y', labelsize=8)
ax.set_xlim(0, top.values.max() * 1.15)
add_panel_label(ax, 'E')
plt.tight_layout()
save_fig(fig, 'E_gene_families')

# ── Figure F: AMR mechanisms pie ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
mech = df_fs_best['mechanism'].replace('', 'Unclassified').value_counts().head(6)
wedges, texts, autotexts = ax.pie(
    mech.values,
    labels=[m[:35] for m in mech.index],
    autopct='%1.1f%%',
    colors=CB_LIST[:len(mech)],
    startangle=90,
    pctdistance=0.8,
    labeldistance=1.12
)
for text in texts:
    text.set_fontsize(9)
for autotext in autotexts:
    autotext.set_fontsize(8)
ax.set_title('AMR resistance mechanisms
(Foldseek structural hits)', fontsize=11)
add_panel_label(ax, 'F')
plt.tight_layout()
save_fig(fig, 'F_mechanisms')

# ── Figure G: Drug target prioritization ──────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
cats   = ['Has human\nhomolog', 'Priority targets\n(no human homolog)']
values = [len(has_human), len(no_human)]
colors_g = [CB['skyblue'], CB['green']]
bars = ax.bar(cats, values, color=colors_g, edgecolor='white', width=0.5, linewidth=0.8)
for b, n in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
            str(n), ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title('Drug target prioritization\n(human non-homology screening)', fontsize=11)
ax.set_ylim(0, max(values) * 1.18)
add_panel_label(ax, 'G')
plt.tight_layout()
save_fig(fig, 'G_drug_targets')

# ── Figure H: E-value distribution ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_fs_best['evalue'], bins=40, color=CB['orange'], edgecolor='white',
        alpha=0.85, linewidth=0.5)
ax.axvline(CONFIG['evalue_threshold'], color='black', linestyle='--', lw=1.5,
           label=f'Filter threshold (e={CONFIG["evalue_threshold"]:.0e})')
ax.set_xlabel('E-value', fontsize=10)
ax.set_ylabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title('E-value distribution of Foldseek structural hits', fontsize=11)
ax.legend(fontsize=9)
add_panel_label(ax, 'H')
plt.tight_layout()
save_fig(fig, 'H_evalue_distribution')

# ── Figure I: KEGG pathways ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
if pw_counter:
    top9 = [(pw.split(':', 1)[-1][:45], n)
            for pw, n in pw_counter.most_common(10) if pw]
    if top9:
        lbls9, cnts9 = zip(*top9)
        bars = ax.barh([truncate_label(l, 45) for l in lbls9[::-1]],
                       cnts9[::-1], color=CB['yellow'], edgecolor='gray',
                       linewidth=0.5, alpha=0.9)
        for b, n in zip(bars, cnts9[::-1]):
            ax.text(b.get_width() + 0.1, b.get_y() + b.get_height()/2,
                    str(n), va='center', fontsize=8)
        ax.set_xlabel('Number of NEIG1 proteins', fontsize=10)
ax.set_title(f'Top KEGG pathways (N. gonorrhoeae FA 1090, {len(kegg_map)} proteins mapped)', fontsize=11)
ax.tick_params(axis='y', labelsize=8)
add_panel_label(ax, 'I')
plt.tight_layout()
save_fig(fig, 'I_kegg_pathways')

# ── Figure J: ESM2 null distribution ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
null_sims = np.load(f'{CONFIG["local_results_dir"]}/esm2_null_sims.npy')
ax.hist(null_sims, bins=50, color=CB['skyblue'], edgecolor='white',
        alpha=0.85, linewidth=0.5, density=True, label='Null distribution\n(NEIG1 random pairs)')
ax.axvline(CONFIG['esm2_threshold'], color=CB['red'], linestyle='--', lw=2,
           label=f'Threshold: {CONFIG["esm2_threshold"]:.4f}\n(p={CONFIG["esm2_pvalue"]:.0e})')
ax.set_xlabel('ESM2 cosine similarity', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('ESM2 null distribution\n(statistical calibration of similarity threshold)', fontsize=11)
ax.legend(fontsize=9)
add_panel_label(ax, 'J')
plt.tight_layout()
save_fig(fig, 'J_esm2_null_distribution')

# ── Figure K: Coverage-identity heatmap ───────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
import pandas as pd
pident_bins   = [0, 30, 50, 70, 90, 100]
pident_labels = ['<30%', '30-50%', '50-70%', '70-90%', '>90%']
tcov_bins     = [0, 0.3, 0.5, 0.7, 0.9, 1.01]
tcov_labels   = ['<30%', '30-50%', '50-70%', '70-90%', '90-100%']

df_fs_best['pid_cat'] = pd.cut(df_fs_best['pident'], bins=pident_bins, labels=pident_labels, right=True)
df_fs_best['tcv_cat'] = pd.cut(df_fs_best['tcov'],   bins=tcov_bins,   labels=tcov_labels,  right=True)
mat = df_fs_best.groupby(['pid_cat','tcv_cat'], observed=True).size().unstack(fill_value=0)

im = ax.imshow(mat.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Number of proteins')
ax.set_xticks(range(len(tcov_labels)))
ax.set_xticklabels(tcov_labels, fontsize=8)
ax.set_yticks(range(len(pident_labels)))
ax.set_yticklabels(pident_labels, fontsize=8)
ax.set_xlabel('Target coverage (tcov)', fontsize=10)
ax.set_ylabel('Sequence identity to AMR query', fontsize=10)
ax.set_title('Structural hit landscape: coverage vs sequence identity\n(Foldseek hits, N. gonorrhoeae FA 1090)', fontsize=11)
# Annotate cells
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat.values[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=9, color='black' if val < mat.values.max()*0.6 else 'white')
add_panel_label(ax, 'K')
plt.tight_layout()
save_fig(fig, 'K_coverage_identity_heatmap')

# ── Figure L: UMAP — AMR embedding space ──────────────────────────────────
df_umap = pd.read_csv(f'{RESULTS}/umap_embedding.tsv', sep='\t')
df_amr_u   = df_umap[df_umap['source'] == 'AMR_query']
df_neig1_u = df_umap[df_umap['source'] == 'NEIG1_hit']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
color_fields = ['mechanism', 'drug_class', 'gene_family']
titles = ['Colored by resistance mechanism',
          'Colored by drug class',
          'Colored by AMR gene family']

for ax, field, title in zip(axes, color_fields, titles):
    cats = df_amr_u[field].fillna('Unknown').value_counts().head(12).index.tolist()
    palette = {cat: CB_LIST[i % len(CB_LIST)] for i, cat in enumerate(cats)}

    # AMR queries (background, small dots)
    for cat in cats:
        sub = df_amr_u[df_amr_u[field].fillna('Unknown') == cat]
        ax.scatter(sub['umap_x'], sub['umap_y'],
                   c=palette[cat], s=3, alpha=0.4, label=cat[:25], rasterized=True)

    # NEIG1 hits (foreground, larger markers, black border)
    ax.scatter(df_neig1_u['umap_x'], df_neig1_u['umap_y'],
               c=CB['red'], s=40, alpha=0.9, zorder=5,
               edgecolors='black', linewidths=0.5,
               label='NEIG1 structural hits', marker='*')

    ax.set_xlabel('UMAP 1', fontsize=9)
    ax.set_ylabel('UMAP 2', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=7)
    # Legend — top categories only
    handles, labels = ax.get_legend_handles_labels()
    # Put NEIG1 hits first
    neig1_h = [h for h, l in zip(handles, labels) if 'NEIG1' in l]
    neig1_l = [l for l in labels if 'NEIG1' in l]
    other_h = [h for h, l in zip(handles, labels) if 'NEIG1' not in l][:8]
    other_l = [l for l in labels if 'NEIG1' not in l][:8]
    ax.legend(neig1_h + other_h, neig1_l + other_l,
              fontsize=6, markerscale=2, loc='best',
              framealpha=0.7, ncol=1)

fig.suptitle(
    f'ESM2 embedding space of AMR query proteins (n={len(df_amr_u):,}) '
    f'with N. gonorrhoeae structural hits overlaid (n={len(df_neig1_u)}, stars)\n'
    f'UMAP: cosine metric, n_neighbors=15, min_dist=0.1, seed={CONFIG["random_seed"]}',
    fontsize=10)
add_panel_label(axes[0], 'L')
plt.tight_layout()
save_fig(fig, 'L_umap_amr_space')

# ── Figure M: PDB TM-score distribution ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_pdb['tmscore'], bins=25, color=CB['blue'], edgecolor='white',
        alpha=0.85, linewidth=0.5)
ax.axvline(0.5, color=CB['orange'], linestyle='--', lw=1.5, label='TM-score 0.5 (same fold)')
ax.axvline(0.7, color=CB['red'],    linestyle='--', lw=1.5, label='TM-score 0.7 (similar struct.)')
ax.set_xlabel('TM-score', fontsize=10)
ax.set_ylabel('Number of hits', fontsize=10)
ax.set_title(f'PDB structural validation: TM-score distribution\n({n_pdb_dl} AMR templates vs NEIG1 AF2 structures)', fontsize=11)
ax.legend(fontsize=9)
add_panel_label(ax, 'M')
plt.tight_layout()
save_fig(fig, 'M_pdb_tmscore')


# ── Figure N: GO enrichment ───────────────────────────────────────────────
if 'df_go_sig' in dir() and len(df_go_sig) > 0:
    df_go_bp = df_go_sig[df_go_sig['namespace'] == 'BP'].head(15)
    if len(df_go_bp) == 0:
        df_go_bp = df_go_sig.head(15)

    if len(df_go_bp) > 0:
        fig, ax = plt.subplots(figsize=(11, 6))
        labels = [r['go_term_clean'][:55] for _, r in df_go_bp.iterrows()]
        folds  = df_go_bp['fold_enrichment'].values
        fdrs   = -np.log10(df_go_bp['fdr'].clip(lower=1e-300).values)

        bars = ax.barh(labels[::-1], folds[::-1],
                       color=CB['green'], edgecolor='white', linewidth=0.5,
                       alpha=0.85)
        ax2 = ax.twiny()
        ax2.scatter(fdrs[::-1], range(len(fdrs)),
                    color=CB['red'], s=50, zorder=5, label='-log10(FDR)')
        ax.set_xlabel('Fold enrichment vs background proteome', fontsize=10)
        ax2.set_xlabel('-log10(FDR)', fontsize=10, color=CB['red'])
        ax.tick_params(axis='y', labelsize=8)
        ax2.tick_params(axis='x', colors=CB['red'], labelsize=8)
        ax.set_title(
            f'GO biological process enrichment\n'
            f'(Fisher\'s exact test, BH FDR<0.05, n={len(df_go_sig)} significant terms)',
            fontsize=11)
        add_panel_label(ax, 'N')
        plt.tight_layout()
        save_fig(fig, 'N_go_enrichment')
else:
    print('  Skipping Figure N: no significant GO terms')

print(f'\n✓ {len(list(Path(FIGS).glob("*.png")))} individual figures saved')
print('\n✓ Cell 17 complete')


## Cell 18: Combined 9-Panel Figure

In [ ]:
# ── CELL 18: Combined 9-panel figure for paper ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

FIGS    = CONFIG['local_figures_dir']
RESULTS = CONFIG['local_results_dir']

# Select 9 panels for combined figure
panel_files = [
    ('A_method_comparison.png',    'A'),
    ('B_venn_foldseek_diamond.png','B'),
    ('C_sequence_identity.png',    'C'),
    ('D_plddt_distribution.png',   'D'),
    ('E_gene_families.png',        'E'),
    ('F_mechanisms.png',           'F'),
    ('G_drug_targets.png',         'G'),
    ('K_coverage_identity_heatmap.png', 'K'),
    ('M_pdb_tmscore.png',          'M'),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()

for ax, (fname, label) in zip(axes, panel_files):
    fpath = f'{FIGS}/{fname}'
    if Path(fpath).exists():
        img = mpimg.imread(fpath)
        ax.imshow(img)
        ax.axis('off')
        ax.text(0.01, 0.99, label, transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')
    else:
        ax.text(0.5, 0.5, f'{fname}\nnot found', ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.axis('off')

N_COMBINED_DISPLAY = locals().get('N_COMBINED', CONFIG.get('swissprot_amr_bacteria_count', ''))
fig.suptitle(
    f'AMRfold: Structure-informed AMR mining in {CONFIG["organism_name"]}\n'
    f'CARD v{CONFIG["card_version"]} + UniProt SwissProt KW-0046 bacteria ({N_COMBINED_DISPLAY} queries) | '
    f'AFDB v6 | Foldseek+ProstT5 (s={CONFIG["foldseek_sensitivity"]}) | 5-method comparison | seed={CONFIG["random_seed"]}',
    fontsize=12, fontweight='bold')

plt.subplots_adjust(hspace=0.05, wspace=0.05)
combined_path = f'{RESULTS}/amrfold_combined_9panel.png'
fig.savefig(combined_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'✓ Combined figure saved: {combined_path}')
print('\n✓ Cell 18 complete')


## Cell 19: Comprehensive HTML + PDF Report

In [ ]:
# ── CELL 19: HTML + PDF report — all results, every step ─────────────────────
import base64, json
from pathlib import Path
from datetime import datetime

RESULTS = CONFIG['local_results_dir']
FIGS    = CONFIG['local_figures_dir']

def img_b64(path):
    try:
        with open(path, 'rb') as f:
            return base64.b64encode(f.read()).decode()
    except:
        return ''

def fig_html(fname, caption='', width='100%'):
    b64 = img_b64(f'{FIGS}/{fname}')
    if not b64: return f'<p style="color:red">Figure {fname} not found</p>'
    return f'''<figure>
    <img src="data:image/png;base64,{b64}" style="width:{width};border-radius:6px;">
    <figcaption style="font-size:0.85rem;color:var(--muted);margin-top:0.5rem">{caption}</figcaption>
</figure>'''

# Load provenance
with open(f'{RESULTS}/provenance.json') as f:
    prov = json.load(f)

# Top hits table
top_hits = df_fs_best.sort_values('evalue').head(30)
rows_html = ''
for _, r in top_hits.iterrows():
    ctype = 'Cryptic' if r['cryptic'] else 'Known'
    ctag  = 'r' if r['cryptic'] else 'g'
    human = 'Priority' if not r['has_human'] else 'Homolog'
    htag  = 'y' if not r['has_human'] else ''
    rows_html += f'''<tr>
      <td><code>{r['neig1_id']}</code></td>
      <td>{r.get('uniprot_gene','')}</td>
      <td>{str(r['short_name'])[:28]}</td>
      <td>{str(r['gene_family'])[:40]}</td>
      <td>{str(r['mechanism'])[:25]}</td>
      <td>{r['pident']:.1f}%</td>
      <td>{r['evalue']:.1e}</td>
      <td>{r.get('mean_plddt',0):.0f}</td>
      <td>{r['tcov']:.2f}</td>
      <td><span class="badge {ctag}">{ctype}</span></td>
      <td><span class="badge {htag}">{human}</span></td>
      <td>{r.get('query_source','')}</td>
    </tr>'''

html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<title>AMRfold v3.0 Report — {CONFIG['organism_name']}</title>
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600;700&display=swap');
:root{{--bg:#0d1117;--surface:#161b22;--border:#30363d;--text:#e6edf3;
      --muted:#8b949e;--accent:#58a6ff;--green:#3fb950;--red:#f85149;
      --yellow:#d29922;--purple:#bc8cff;--orange:#e69f00;}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{background:var(--bg);color:var(--text);font-family:'IBM Plex Sans',sans-serif;
     line-height:1.6;padding:2rem;max-width:1400px;margin:0 auto}}
header{{border-bottom:2px solid var(--border);padding-bottom:1.5rem;margin-bottom:2.5rem}}
header h1{{font-size:2rem;font-weight:700;
          background:linear-gradient(90deg,var(--accent),var(--purple));
          -webkit-background-clip:text;-webkit-text-fill-color:transparent}}
header .meta{{color:var(--muted);font-size:0.88rem;margin-top:0.4rem}}
.section{{margin:2.5rem 0;padding-top:1rem;border-top:1px solid var(--border)}}
h2{{font-size:1.2rem;font-weight:700;color:var(--accent);margin-bottom:1rem}}
h3{{font-size:1rem;font-weight:600;color:var(--purple);margin:1rem 0 0.5rem}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:1rem;margin:1.5rem 0}}
.card{{background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:1rem 1.2rem}}
.card .val{{font-size:1.8rem;font-weight:700;font-family:'IBM Plex Mono',monospace;color:var(--accent)}}
.card .label{{font-size:0.77rem;color:var(--muted);margin-top:0.2rem;line-height:1.3}}
.card.g .val{{color:var(--green)}}.card.r .val{{color:var(--red)}}
.card.y .val{{color:var(--yellow)}}.card.p .val{{color:var(--purple)}}
.card.o .val{{color:var(--orange)}}
table{{width:100%;border-collapse:collapse;font-size:0.82rem;margin:1rem 0}}
th{{background:var(--surface);color:var(--muted);text-align:left;
    padding:0.6rem 0.7rem;font-weight:600;position:sticky;top:0}}
td{{padding:0.45rem 0.7rem;border-bottom:1px solid var(--border)}}
tr:hover td{{background:var(--surface)}}
code{{font-family:'IBM Plex Mono',monospace;font-size:0.82em;color:var(--purple)}}
.badge{{display:inline-block;padding:0.1rem 0.4rem;border-radius:4px;
        font-size:0.75rem;font-weight:600}}
.badge.g{{background:rgba(63,185,80,0.2);color:var(--green)}}
.badge.r{{background:rgba(248,81,73,0.2);color:var(--red)}}
.badge.y{{background:rgba(210,153,34,0.2);color:var(--yellow)}}
.lim{{background:var(--surface);border-left:3px solid var(--yellow);
      border-radius:0 6px 6px 0;padding:1rem 1.2rem}}
.lim li{{margin:0.4rem 0;color:var(--muted);font-size:0.88rem}}
figure{{margin:1rem 0}}
figcaption{{text-align:center}}
pre{{background:var(--surface);border:1px solid var(--border);border-radius:6px;
     padding:1rem;font-family:'IBM Plex Mono',monospace;font-size:0.8rem;
     overflow-x:auto;white-space:pre-wrap}}
footer{{margin-top:4rem;border-top:1px solid var(--border);padding-top:1rem;
        color:var(--muted);font-size:0.8rem;text-align:center}}
.fig-grid{{display:grid;grid-template-columns:1fr 1fr;gap:1rem}}
</style></head><body>

<header>
<h1>AMRfold v3.0 — Analysis Report</h1>
<div class="meta">
<em>{CONFIG['organism_name']}</em> |
CARD v{CONFIG['card_version']} + UniProt SwissProt KW-0046 bacteria ({prov['n_combined_nr']:,} queries) |
AFDB v6 | Foldseek+ProstT5 (s={CONFIG['foldseek_sensitivity']}) |
Seed: {CONFIG['random_seed']} |
{datetime.now().strftime('%Y-%m-%d %H:%M')}
</div>
</header>

<!-- 0: RUN CONFIGURATION -->
<div class="section">
<h2>0. Run Configuration</h2>
<pre>{json.dumps({{k:v for k,v in CONFIG.items() if k not in ['tool_versions']}}, indent=2, default=str)}</pre>
<h3>Tool Versions</h3>
<pre>{json.dumps(prov.get('tool_versions',{{}}), indent=2)}</pre>
</div>

<!-- 1: DATABASE STATISTICS -->
<div class="section">
<h2>1. Database Statistics</h2>
<div class="grid">
  <div class="card"><div class="val">{prov['n_card_sequences']:,}</div><div class="label">CARD protein<br>homolog sequences</div></div>
  <div class="card"><div class="val">{prov['n_swissprot_sequences']:,}</div><div class="label">UniProt SwissProt<br>KW-0046 bacteria</div></div>
  <div class="card g"><div class="val">{prov['n_combined_nr']:,}</div><div class="label">Combined non-redundant<br>(90% clustering)</div></div>
  <div class="card"><div class="val">{prov['n_structures']:,}</div><div class="label">NEIG1 AF2<br>structures</div></div>
</div>
<h3>Cluster Membership</h3>
<pre>{json.dumps(prov.get('cluster_stats',{{}}), indent=2)}</pre>
</div>

<!-- 2: KEY RESULTS -->
<div class="section">
<h2>2. Key Results</h2>
<div class="grid">
  <div class="card"><div class="val">{prov['results']['foldseek']}</div><div class="label">Structural AMR hits<br>({prov['results']['foldseek']/prov['n_structures']*100:.1f}% proteome)</div></div>
  <div class="card r"><div class="val">{prov['results']['cryptic']}</div><div class="label">Cryptic hits<br>(&lt;30% seq. identity)</div></div>
  <div class="card p"><div class="val">{prov['results']['foldseek']/max(prov['results']['diamond'],1):.1f}x</div><div class="label">Gain over DIAMOND</div></div>
  <div class="card"><div class="val">{prov['results']['exclusive_vs_4method']}</div><div class="label">Exclusive vs<br>all 4 methods</div></div>
  <div class="card"><div class="val">{prov['results']['exclusive_vs_5method']}</div><div class="label">Exclusive vs<br>all 5 methods</div></div>
  <div class="card g"><div class="val">{prov['results']['high_plddt_70']}</div><div class="label">High pLDDT &gt;70</div></div>
  <div class="card y"><div class="val">{prov['results']['no_human_homolog']}</div><div class="label">Priority drug targets<br>(no human homolog)</div></div>
  <div class="card o"><div class="val">{prov['results']['multidomain_proteins']}</div><div class="label">Multi-domain<br>AMR proteins</div></div>
</div>
</div>

<!-- 3: FIVE-WAY COMPARISON -->
<div class="section">
<h2>3. Five-Method Comparison</h2>
{fig_html('A_method_comparison.png', 'Figure A: AMR detection by search method', '70%')}
<table>
<tr><th>Method</th><th>Hits</th><th>% Proteome</th><th>vs Foldseek</th><th>Database</th></tr>
<tr><td>MMseqs2 (sequence alignment)</td><td>{prov['results']['mmseqs2']}</td><td>{prov['results']['mmseqs2']/prov['n_structures']*100:.1f}%</td><td>{prov['results']['foldseek']/max(prov['results']['mmseqs2'],1):.1f}x fewer</td><td>Combined protein query</td></tr>
<tr><td>DIAMOND BLASTp (sensitive)</td><td>{prov['results']['diamond']}</td><td>{prov['results']['diamond']/prov['n_structures']*100:.1f}%</td><td>{prov['results']['foldseek']/max(prov['results']['diamond'],1):.1f}x fewer</td><td>Combined protein query</td></tr>
<tr><td>ESM2 pLM (stat. calibrated, p=1e-5)</td><td>{prov['results']['esm2']}</td><td>{prov['results']['esm2']/prov['n_structures']*100:.1f}%</td><td>{prov['results']['foldseek']/max(prov['results']['esm2'],1):.1f}x fewer</td><td>Combined protein query</td></tr>
<tr><td>AMRFinderPlus HMM (BLAST+HMM)</td><td>{prov['results']['amrfinderplus_full']}</td><td>{prov['results']['amrfinderplus_full']/prov['n_structures']*100:.1f}%</td><td>{prov['results']['foldseek']/max(prov['results']['amrfinderplus_full'],1):.1f}x fewer</td><td>NCBI independent DB</td></tr>
<tr style="color:var(--accent);font-weight:700"><td>Foldseek+ProstT5 [AMRfold]</td><td>{prov['results']['foldseek']}</td><td>{prov['results']['foldseek']/prov['n_structures']*100:.1f}%</td><td>—</td><td>Combined protein query</td></tr>
</table>
<div class="fig-grid">
{fig_html('B_venn_foldseek_diamond.png', 'Figure B: Foldseek vs DIAMOND overlap')}
{fig_html('C_sequence_identity.png', 'Figure C: Sequence identity distribution')}
</div>
</div>

<!-- 4: STRUCTURAL HIT ANALYSIS -->
<div class="section">
<h2>4. Structural Hit Analysis</h2>
<div class="fig-grid">
{fig_html('K_coverage_identity_heatmap.png', 'Figure K: Coverage-identity structural hit landscape')}
{fig_html('D_plddt_distribution.png', 'Figure D: AlphaFold2 model confidence (pLDDT)')}
</div>
<h3>PDB Structural Validation</h3>
<pre>{json.dumps(prov.get('pdb_validation',{{}}), indent=2)}</pre>
{fig_html('M_pdb_tmscore.png', 'Figure M: TM-score distribution (25 AMR PDB templates)', '60%')}
</div>

<!-- 5: AMR ANNOTATION -->
<div class="section">
<h2>5. AMR Annotation</h2>
<div class="fig-grid">
{fig_html('E_gene_families.png', 'Figure E: Top AMR gene families')}
{fig_html('F_mechanisms.png', 'Figure F: Resistance mechanisms')}
</div>
</div>

<!-- 6: DRUG TARGET PRIORITIZATION -->
<div class="section">
<h2>6. Drug Target Prioritization</h2>
{fig_html('G_drug_targets.png', 'Figure G: Human non-homology screening', '50%')}
</div>

<!-- 7: PATHWAY ANALYSIS -->
<div class="section">
<h2>7. KEGG Pathway Analysis</h2>
{fig_html('I_kegg_pathways.png', 'Figure I: Top KEGG pathways')}
</div>

<!-- 8: ESM2 ANALYSIS -->
<div class="section">
<h2>8. ESM2 Statistical Calibration + AMR Embedding Space</h2>
<pre>Null distribution: {prov.get('esm2_null_distribution','')}
Mean: {prov.get('esm2_null_mean',0):.4f} | Std: {prov.get('esm2_null_std',0):.4f}
Threshold (p={prov.get('esm2_pvalue',1e-5):.0e}): {prov.get('esm2_threshold',0):.4f}
Silhouette (by mechanism): {prov.get('esm2_silhouette',None)}</pre>
{fig_html('J_esm2_null_distribution.png', 'Figure J: ESM2 null distribution and statistical threshold', '60%')}
{fig_html('L_umap_amr_space.png', 'Figure L: UMAP of AMR protein embedding space with NEIG1 hits overlaid (stars)')}
</div>

<!-- 9: ALL HITS TABLE -->
<div class="section">
<h2>9. Top 30 Structural AMR Hits</h2>
<div style="overflow-x:auto">
<table>
<tr><th>UniProt ID</th><th>Gene</th><th>CARD Hit</th><th>AMR Family</th><th>Mechanism</th>
<th>Identity</th><th>E-value</th><th>pLDDT</th><th>tcov</th><th>Type</th><th>Target</th><th>Source</th></tr>
{rows_html}
</table></div>
</div>

<!-- 10: LIMITATIONS -->
<div class="section">
<h2>10. Limitations</h2>
<div class="lim"><ul>
{''.join(f"<li>{l}</li>" for l in prov.get('limitations',[]))}
</ul></div>
</div>

<!-- 11: PROVENANCE -->
<div class="section">
<h2>11. Complete Provenance</h2>
<pre>{json.dumps(prov, indent=2, default=str)}</pre>
</div>

<footer>
AMRfold v3.0 | github.com/pranavathiyani/amrfold | SASTRA Deemed University | April 2026<br>
CARD v{CONFIG['card_version']} | UniProt SwissProt KW-0046 bacteria | AFDB v6 | Seed: {CONFIG['random_seed']}
</footer>
</body></html>'''

report_path = f'{RESULTS}/amrfold_report.html'
with open(report_path, 'w') as f:
    f.write(html)
print(f'✓ HTML report: {report_path}')

# ── PDF via weasyprint ─────────────────────────────────────────────────────
try:
    !pip install -q weasyprint 2>/dev/null
    from weasyprint import HTML as WHTML
    pdf_path = f'{RESULTS}/amrfold_report.pdf'
    WHTML(filename=report_path).write_pdf(pdf_path)
    print(f'✓ PDF report: {pdf_path}')
except Exception as e:
    print(f'  PDF generation skipped: {e}')
    print('  (HTML report is complete — open in browser)')

print('\n✓ Cell 19 complete')


## Cell 20: Download All Outputs

In [ ]:
# ── CELL 20: Download all outputs ─────────────────────────────────────────────
from google.colab import files
from pathlib import Path

RESULTS = CONFIG['local_results_dir']
FIGS    = CONFIG['local_figures_dir']

outputs = [
    # Main report
    (f'{RESULTS}/amrfold_report.html',          'HTML report (open in browser)'),
    (f'{RESULTS}/amrfold_report.pdf',            'PDF report'),
    # Data
    (f'{RESULTS}/neig1_amr_best.tsv',            'Best hit per NEIG1 protein'),
    (f'{RESULTS}/neig1_amr_all.tsv',             'All hits (multi-domain analysis)'),
    (f'{RESULTS}/neig1_amr_all_domains.tsv',     'Multi-domain analysis'),
    (f'{RESULTS}/cluster_membership.tsv',         'Query DB cluster membership'),
    (f'{RESULTS}/amrfinder_out.tsv',             'AMRFinderPlus raw output'),
    (f'{RESULTS}/umap_embedding.tsv',            'UMAP coordinates'),
    (f'{RESULTS}/kegg_pathways.json',            'KEGG pathway mappings'),
    (f'{RESULTS}/uniprot_cache.json',            'UniProt annotations cache'),
    (f'{RESULTS}/provenance.json',               'Complete provenance metadata'),
    # Figures
    (f'{RESULTS}/amrfold_combined_9panel.png',   'Combined 9-panel figure'),
]

# Add individual figures
for fig_file in sorted(Path(FIGS).glob('*.png')):
    outputs.append((str(fig_file), f'Figure: {fig_file.stem}'))

print(f'Downloading {len(outputs)} files...\n')
downloaded, missing = 0, 0
for path, desc in outputs:
    if Path(path).exists():
        files.download(path)
        size = Path(path).stat().st_size / 1024
        print(f'  ✓ {desc} ({size:.0f} KB)')
        downloaded += 1
    else:
        print(f'  ○ {desc} — not found (skipped)')
        missing += 1

print(f'\n🏁 AMRfold v3.0 complete!')
print(f'   Downloaded: {downloaded} | Skipped: {missing}')
print(f'   Foldseek: {len(fold_ids)} hits | Cryptic: {df_fs_best["cryptic"].sum()} | Priority: {len(no_human)}')
print(f'   Seed: {CONFIG["random_seed"]} | {datetime.now().strftime("%Y-%m-%d %H:%M")}')
